## About this notebook

We developed a query for PubMed, Scopus and Web of Science that optimizes the number of relevant sources it finds against the number of records it returns. This notebook has the code, results and files for each test we ran to develop the search. The [search strategy](../../materials/search-strategy.qmd) page summarizes the method, the results and their limits, and `log.md` records each decision with its date.

In [1]:
import math
import re
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").exists())
RAW = ROOT / "data" / "raw" / "search-development"
PROCESSED = ROOT / "data" / "processed" / "search-development"
PROCESSED_FILES = {"reference_sources.csv", "terms_counts.csv", "factor_term_counts.csv"}
DATABASES = {"PubMed": "pubmed", "Scopus": "scopus", "Web of Science": "web-of-science"}


def path(name):
    return (PROCESSED if name in PROCESSED_FILES else RAW) / name


def read(name):
    return pd.read_csv(path(name), dtype=str, keep_default_na=False)



In [2]:
#| output: false
cand = read("reference_candidates.csv")
look = read("reference_lookups.csv")


def write(df, name):
    PROCESSED.mkdir(parents=True, exist_ok=True)
    df.to_csv(PROCESSED / name, index=False, lineterminator="\n")


# the 102 sources with their bibliographic details
src = look[["ref_id", "title", "pub_date", "doi", "pmid", "abstract_source"]].merge(
    cand[["ref_id", "label", "cited_in"]], on="ref_id")
src["year"] = src["pub_date"].str[:4]
src = src[["ref_id", "label", "title", "year", "pub_date", "doi", "pmid", "cited_in", "abstract_source"]].sort_values("ref_id")
write(src, "reference_sources.csv")
print(f"Built reference_sources.csv ({len(src)} sources).")


Built reference_sources.csv (102 sources).


## Known sources

We measured each candidate query by how many known relevant sources it finds ([Hausner et al. 2012](https://doi.org/10.1186/2046-4053-1-19)). The known sources are the 102 sources included in [Chawinga and Zinn's (2019)](https://doi.org/10.1016/j.lisr.2019.04.004) review of research data sharing.

### Run 1. Rebuilding Chawinga and Zinn's list

Chawinga and Zinn did not publish the list of their 105 sources included in their review, and it was therefore necessary to rebuild the list from the paper's full text (using [`retrieval/reference_set.py`](https://github.com/zackbatist/RDS-Review/blob/main/analysis/00_search-development/retrieval/reference_set.py)). We took the references from Table 2 and the references cited in sections 4 and 5. For each country section 4.3 lists that no other selected reference names, we also took the reference from 2007 to 2018 whose title names it. This produced a set of 102 sources. With [`retrieval/lookups.py`](https://github.com/zackbatist/RDS-Review/blob/main/analysis/00_search-development/retrieval/lookups.py), we checked each DOI against Crossref and OpenAlex and pulled the PMID, the earliest publication date, the title, the abstract, the MeSH headings and the keywords.

Files: [reference_candidates.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/reference_candidates.csv), [reference_lookups.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/reference_lookups.csv).

## Terms the known sources use

In runs 2 and 3, we read Chawinga and Zinn's sources for the words and phrases that describe research data sharing. Run 2 uses a simple, open-ended prompt, and run 3 matches the specific terms documented in the screening guide. We ran them on 23 November 2026.

### Run 2. Terms the sources use

**Question.** Which words do the sources included in Chawinga and Zinn use for research data sharing?

**Measured.** A local model, Qwen3.8-27B, read the title and abstract of each of the 102 sources in the reference set with [`prompts/terms_prompt.md`](https://github.com/zackbatist/RDS-Review/blob/main/analysis/00_search-development/prompts/terms_prompt.md) and listed the words and phrases the source uses to describe research data sharing, copied exactly. The prompt does not name any of the specific terms mentioned in the screening guide. It reads:

> You are helping build a literature search for a scoping review on research data sharing. Research data are data collected for a study, or collected for another purpose and reused for research.
>
> From the title and abstract below, list the words and phrases the source uses to describe research data sharing. Copy each term exactly as it appears. Each term is a word or short phrase that names one thing. Give an empty list if there are none.
>
> Answer only with JSON: `{"terms": ["<term>"]}`
>
> Title: {title}\
> Abstract: {abstract}

We dropped terms that are not in the source's title or abstract, joined terms that differ only in case or hyphens, and counted the sources that use each term.

**Result.** The model returned 412 terms, 259 of them distinct; 29 appear in two or more sources, and 18 sources yield no term. The recurring terms fall into six patterns:

- **The same idea, worded differently:** "data sharing" (50 sources), "sharing data" (12), "share data" (11), "sharing of data" (4), "shared data" (2), "share their data" (2).
- **Sharing with the kind of data named:** "sharing research data" (4), "research data sharing" (3), "biodiversity data sharing" (3), "scientific data sharing" (2), "sharing individual level data" (2).
- **Sharing as a policy or practice:** "data sharing policies" (5), "data sharing practices" (3), "data sharing policy" (2), "data sharing program" (2), "data management and sharing practices" (2), "data sharers" (2).
- **Access without *share*:** "open data" (3), "data accessibility" (2), "making research data accessible" (2), "publish data" (2), "data repository" (2).
- **Management and curation:** "data curation" (3), "data management" (3), "data management plans" (3), "research data management services" (2).
- **Reuse:** "data reuse" (2).

The first three patterns hold *share* and *data* within a few words of each other. Each term in the last three appears in two or three sources.

Files: [terms_counts.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/processed/search-development/terms_counts.csv), [terms_qwen3.8-27b.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/terms_qwen3.8-27b.csv), [terms_console_qwen3.8-27b.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/terms_console_qwen3.8-27b.txt).

In [5]:
# On the analysis server, with the local model running (scripts/launch_rds_llm.sh):
#   LLM_URL=http://<GPU server>:11451 python3 analysis/00_search-development/local-model/terms.py terms_qwen3.8-27b.csv
if path("terms_qwen3.8-27b.csv").exists():
    term_rows = read("terms_qwen3.8-27b.csv")
    ids = read("reference_sources.csv")["ref_id"]
    found = term_rows[term_rows["location"] != "not in text"].copy()
    found["term"] = found["term"].str.lower().str.replace(r"[-‐–]", " ", regex=True).str.split().str.join(" ")
    found = found.drop_duplicates(["ref_id", "term"])
    table = (found.groupby("term").size().rename("sources").reset_index()
             .sort_values(["sources", "term"], ascending=[False, True]).set_index("term"))
    print(f"{len(term_rows)} terms, {len(table)} distinct, {int((table['sources'] > 1).sum())} used by two or more sources; "
          f"{int((term_rows['location'] == 'not in text').sum())} not in the text")
    none = sorted(set(ids) - set(term_rows["ref_id"]))
    print(f"{len(none)} sources with no term")
    write(table.reset_index(), "terms_counts.csv")
else:
    print("Not run yet: terms_qwen3.8-27b.csv is missing.")

412 terms, 259 distinct, 29 used by two or more sources; 10 not in the text
18 sources with no term


### Run 3. Factor terms in the sources

**Question.** How often do the terms in the screening guide's factors table appear in the titles and abstracts of Chawinga and Zinn's 102 sources, and how do they spread across the sources?

**Measured.** We matched each factor term against each source's title and abstract. A term ending in \* matches any word that starts with it, and a quoted term matches as a phrase. We counted the sources that use each term and each theme, and the number of themes each source touches.

**Result.** 79 sources use at least one factor term and 23 use none; 12 of those 23 have a title only. *polic\** (28 sources) and *challenge\** (18) are the terms used most, and *facilitator\**, *interoperab\**, *"open research"* and *licens\** appear in no source. *"research data management"* appears in 14 sources.

Files: [factor_term_counts.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/processed/search-development/factor_term_counts.csv), [reference_lookups.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/reference_lookups.csv).

In [6]:
TABLE1 = {
    "factors that obstruct": ["barrier*", "challenge*", "obstacle*", "reluctan*", "privacy"],
    "factors that facilitate": ["facilitator*", "enabler*", "incentive*", "motivat*", "infrastructure", "repositor*",
                                "interoperab*"],
    "stakeholder perspectives": ["perception*", "attitude*", "willingness"],
    "policies and what drive them": ["polic*", "mandate*", '"open science"', '"open data"', '"open access"',
                                     '"open research"'],
    "mechanisms for implementation": ["governance", "consent", "licens*", "ethic*", "stewardship",
                                      '"data management plan*"', '"data curation"', '"digital curation"',
                                      '"research data management"'],
}


def pattern(term):
    """A Table 1 term as a regular expression: * ends a word stem, quotes mark a phrase."""
    words = term.strip('"').split()
    parts = [re.escape(w[:-1]) + r"\w*" if w.endswith("*") else re.escape(w) for w in words]
    return re.compile(r"\b" + r"\s+".join(parts) + r"\b")


def plain(text):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", text.lower())).strip()


if path("reference_lookups.csv").exists():
    lookups = read("reference_lookups.csv").set_index("ref_id")
    text = (lookups["title"] + " " + lookups["abstract"]).map(plain)
    hits = pd.DataFrame({(theme, term): text.map(lambda t, p=pattern(term): bool(p.search(t)))
                         for theme, terms in TABLE1.items() for term in terms})
    hits.columns.names = ["theme", "term"]
    def counts(frame):
        return frame.sum().astype(int).rename("sources").to_frame()

    by_term = counts(hits)
    themes = hits.T.groupby(level="theme").any().T
    by_theme = counts(themes).reindex(TABLE1)
    per_source = themes.sum(axis=1)
    coded_on = lookups["abstract"].map(lambda a: "title and abstract" if a.strip() else "title only").rename("coded on")
    spread = pd.crosstab(per_source.rename("themes per source"), coded_on)
    print(f"{int((per_source > 0).sum())} sources use a factor term, {int((per_source == 0).sum())} use none")
    display(by_theme)
    write(by_term.reset_index(), "factor_term_counts.csv")
else:
    print("Not run yet: reference_lookups.csv is missing.")

79 sources use a factor term, 23 use none


,sources
theme,
factors that obstruct,33
factors that facilitate,29
stakeholder perspectives,14
policies and what drive them,39
mechanisms for implementation,42


## Open data and research data management

Runs 4 and 5 concern phrases beside *shar\** near data, and run 6 concerns the query's records and the known sources among them. All searches are in titles and abstracts, on 25 November 2026.

In [9]:
#| eval: false
# Setup for every run below.
import re
import sys
import itertools

sys.path.insert(0, str(ROOT / "analysis" / "00_search-development" / "retrieval"))
import search_lib as sl

sl.BATCH = 10  # identifiers per request when checking which known sources a query finds

LANG = {"PubMed": "(english[la] OR french[la])", "Scopus": "LANGUAGE(english OR french)",
        "Web of Science": "LA=(English OR French)"}
TAG = {"title": {"PubMed": "{}[ti]", "Scopus": "TITLE({})", "Web of Science": "TI=({})"},
       "anywhere": {"PubMed": "{}[tiab]", "Scopus": "TITLE-ABS-KEY({})", "Web of Science": "TS=({})"}}

def limit(db):
    return f"{sl.date_limit(db, 2015, 2025)} AND {LANG[db]}"


def field(db, where, terms):
    """The terms OR-joined in one field: the title, or anywhere in title, abstract and keywords."""
    return "(" + " OR ".join(TAG[where][db].format(t) for t in terms) + ")"


FORMS = ("sharing", "shared", "share", "shares", "sharer", "sharers", "shareable", "sharable")  # PubMed proximity takes no truncation


def share_near_data(db, where, n):
    """A form of share within n words of data, in the title or anywhere. Any word order matches."""
    if db == "PubMed":
        tag = "ti" if where == "title" else "tiab"
        return "(" + " OR ".join(f'"{w} data"[{tag}:~{n}]' for w in FORMS) + ")"
    t = {"Scopus": {"title": "TITLE", "anywhere": "TITLE-ABS-KEY"},
         "Web of Science": {"title": "TI=", "anywhere": "TS="}}[db][where]
    return f"{t}(shar* W/{n} data)" if db == "Scopus" else f"{t}(shar* NEAR/{n} data)"


def limit_near(db):
    """Web of Science's API rejects LA beside shar* NEAR/n data, so there the date limit stands alone."""
    return sl.date_limit(db, 2015, 2025) if db == "Web of Science" else limit(db)


def fetch(db, q, lim):
    """Every record a query returns, with DOI, title and year."""
    full = f"({q}) AND {lim}"
    if db == "PubMed":
        return list(sl.pubmed_summary(sl.pubmed_all_ids(full)).values())
    return sl.scopus_records(full) if db == "Scopus" else sl.wos_records(full)


def unique_count(recs):
    """Records left once those sharing a DOI, or a title and year, count as one. bib-dedupe finds as many or more."""
    seen, unique = {}, 0
    for r in recs:
        keys = [k for k in (("doi", r["doi"]) if r["doi"] else None,
                            ("title", re.sub(r"[^a-z0-9]", "", r["title"].lower()) + r["year"]) if r["title"] else None)
                if k]
        hit = next((seen[k] for k in keys if k in seen), None)
        if hit is None:
            unique += 1
            hit = unique
        for k in keys:
            seen[k] = hit
    return unique


SRC = sl.read("reference_sources.csv")


### Run 4. Phrases beside *shar\** near data

**Question.** What is the effect of adding "research data management", "open science" or "data publication\*" to (shar\* NEAR/5 data) AND (*research\** OR *scienti\**), compared with *open\** NEAR/5 data?

**Measured.** Each phrase joined by OR to (shar\* NEAR/5 data) AND (*research\** OR *scienti\**), alone and all three together, in PubMed, Scopus and Web of Science. In PubMed, the *shar\** part spells out eight forms of *share*. For each search, the records for 2015 to 2025 and the matches among the 102 sources, with no date limit.

**Result.** Each cell gives records and matches among the 102 sources.

| Search | PubMed | Scopus | Web of Science |
|:--|:--|:--|:--|
| (shar\* NEAR/5 data) AND (*research\** OR *scienti\**) | 9,271 · 32 | 31,681 · 52 | 22,770 · 44 |
| … OR "research data management" | 9,415 · 32 | 32,655 · 57 | 23,499 · 49 |
| … OR "open science" | 14,262 · 34 | 40,645 · 55 | 30,524 · 47 |
| … OR "data publication\*" | 9,402 · 32 | 32,471 · 52 | 23,407 · 44 |
| … OR all three | 14,515 · 34 | 42,246 · 60 | 31,776 · 52 |
| (shar\* NEAR/5 data OR *open\** NEAR/5 data) AND (*research\** OR *scienti\**) | 14,409 · 33 | 55,739 · 54 | 40,745 · 46 |

The cost of "research data management" is 1,847 records across the three databases, for six known sources, Burgi et al. (2016), Charbonneau (2013), Chiware and Mathe (2016), Clement et al. (2016), Wittenberg and Elings (2017) and Yoon and Schultz (2017). The cost of "open science" is 21,709 records, for three, Houtkoop et al. (2018), Mennes et al. (2013) and Watson (2015). The cost of "data publication\*" is 1,558 records, for none. The cost of *open\** NEAR/5 data is 47,171 records, for two, McCullough (2009) and Specht et al. (2015). Beside the four verb forms of *share*, the other four forms in PubMed cost 77 records, for none.

**Decision.** The query includes "research data management". "open science" stays out, since it covers matters outside the review such as reproducibility and reuse. "data publication\*" stays out, since it matches no known source. The PubMed string includes *sharer*, *sharers*, *shareable* and *sharable*, so that it covers the same words as *shar\** in Scopus and Web of Science.

In [ ]:
#| eval: false
# Phrases beside shar* near data, against all 102 sources. Prints only.
def rs(db):
    return field(db, "anywhere", ["research*", "scienti*"])


def minus(db, a, b):
    return f"({a} AND NOT {b})" if db == "Scopus" else f"({a} NOT {b})"


def open_near(db):
    if db == "PubMed":
        return "(" + " OR ".join(f'"{w} data"[tiab:~5]' for w in ("open", "opened", "opening", "openly", "openness")) + ")"
    return "TITLE-ABS-KEY(open* W/5 data)" if db == "Scopus" else "TS=(open* NEAR/5 data)"


RDM, OS, DP = '"research data management"', '"open science"', '"data publication*"'


def share_m(db):
    return share_near_data(db, "anywhere", 5)


def base_m(db):
    return f"({share_m(db)} AND {rs(db)})"


def plus(db, terms):
    return f"({base_m(db)} OR {field(db, 'anywhere', terms)})"


S = [("(shar* NEAR/5 data OR open* NEAR/5 data) AND (research* OR scienti*)", lambda db: f"(({share_near_data(db, 'anywhere', 5)} OR {open_near(db)}) AND {rs(db)})"),
     ("(shar* NEAR/5 data) AND (research* OR scienti*)", base_m),
     ('... OR "research data management"', lambda db: plus(db, [RDM])),
     ('... OR "open science"', lambda db: plus(db, [OS])),
     ('... OR "data publication*"', lambda db: plus(db, [DP])),
     ('... OR all three', lambda db: plus(db, [RDM, OS, DP]))]
for db in DATABASES:
    t = [s for s in SRC if sl.testable(db, s)]
    for name, make in S:
        q = make(db)
        print(db, "|", name, "| records", sl.count(db, f"({q}) AND {limit_near(db)}"), "| of the 102 found", len(sl.retrieved(db, q, t)))

### Run 5. "open data" as a phrase

**Question.** Is there a narrower way to include open data than *open\** NEAR/5 data?

**Measured.** "open data" AND (*research\** OR *scienti\**), and "open research data", each joined by OR to (shar\* NEAR/5 data AND (*research\** OR *scienti\**)) OR "research data management", alone and together. For each search, the records for 2015 to 2025, the added records and the matches among the 102 sources.

**Result.** Each cell gives records and matches among the 102 sources.

| Search | PubMed | Scopus | Web of Science |
|:--|:--|:--|:--|
| (shar\* NEAR/5 data AND (*research\** OR *scienti\**)) OR "research data management" | 9,415 · 32 | 32,655 · 57 | 23,499 · 49 |
| … OR ("open data" AND (*research\** OR *scienti\**)) | 10,435 · 33 | 39,442 · 58 | 28,143 · 50 |
| … OR "open research data" | 9,429 · 32 | 32,751 · 57 | 23,584 · 49 |
| … OR both | 10,448 · 33 | 39,498 · 58 | 28,200 · 50 |

The cost of "open data" with the research terms is 12,451 records across the three databases, for one known source, Specht et al. (2015). The cost of "open research data" is 195 records, for none.

**Decision.** The query includes "open data" AND (*research\** OR *scienti\**). A query on research data sharing that omits open data would be incomplete, even though few of the 102 sources use the term. "open research data" stays out, since it matches no known source.

In [ ]:
#| eval: false
# "open data" and "open research data" beside shar* near data and "research data management". Prints only.
def f1(db, t):
    return field(db, "anywhere", [t])


def base_r(db):
    return f"(({share_near_data(db, 'anywhere', 5)} AND {rs(db)}) OR {f1(db, RDM)})"


def od(db):
    return f"({f1(db, chr(34) + 'open data' + chr(34))} AND {rs(db)})"


def ord_(db):
    return f1(db, '"open research data"')


S = [("base", base_r), ('... OR "open data"', lambda db: f"({base_r(db)} OR {od(db)})"),
     ('... OR "open research data"', lambda db: f"({base_r(db)} OR {ord_(db)})"),
     ("... OR both", lambda db: f"({base_r(db)} OR {od(db)} OR {ord_(db)})")]
for db in DATABASES:
    t = [s for s in SRC if sl.testable(db, s)]
    for name, make in S:
        q = make(db)
        added = sl.count(db, f"({minus(db, q, base_r(db))}) AND {limit_near(db)}")
        print(db, "|", name, "| records", sl.count(db, f"({q}) AND {limit_near(db)}"), "| added", added,
              "| of the 102 found", len(sl.retrieved(db, q, t)))

### The query

(shar\* NEAR/5 data OR "open data" OR "research data management") AND (*research\** OR *scienti\**), in titles and abstracts, 2015 to 2025, English or French.

| Database | Query |
|:--|:--|
| PubMed | `(("sharing data"[tiab:~5] OR "shared data"[tiab:~5] OR "share data"[tiab:~5] OR "shares data"[tiab:~5] OR "sharer data"[tiab:~5] OR "sharers data"[tiab:~5] OR "shareable data"[tiab:~5] OR "sharable data"[tiab:~5]) OR ("open data"[tiab] OR "research data management"[tiab])) AND (research*[tiab] OR scienti*[tiab])` |
| Scopus | `(TITLE-ABS-KEY(shar* W/5 data) OR TITLE-ABS-KEY("open data") OR TITLE-ABS-KEY("research data management")) AND (TITLE-ABS-KEY(research*) OR TITLE-ABS-KEY(scienti*))` |
| Web of Science | `(TS=(shar* NEAR/5 data) OR TS=("open data") OR TS=("research data management")) AND (TS=(research*) OR TS=(scienti*))` |

: The query as written for each database. {tbl-colwidths="[16,84]"}

### Run 6. The query against the 102 sources, and its records

**Question.** How many of the 102 sources are in each database, how many match the query, and how many records are in the query's results?

**Measured.** Each of the 102 sources with a DOI or PMID, checked by identifier in each database and against the query, with no date limit. Every record in the query's results for 2015 to 2025, with its DOI, title and year. Records with the same DOI, or the same title and year, count once.

**Result.**

| | Records | Known sources in the database | Matching the query |
|:--|--:|--:|--:|
| PubMed | 10,435 | 43 | 33 |
| Scopus | 39,442 | 70 | 58 |
| Web of Science | 28,143 | 65 | 50 |
| At least one database | 78,020 | 76 | 62 |
| Unique | 42,925 | | |

: Records for 2015 to 2025 and the 102 sources, of which 85 have a DOI or PMID. {tbl-colwidths="[28,24,24,24]"}

Of the sources in a database, 14 do not match the query, Cahill and Passamano (2007), Cohn (2012), Delson et al. (2007), Gupta (2013), Houtkoop et al. (2018), Huang et al. (2013), Kowalczyk and Shankar (2011), Levenson (2010), McCullough (2009), Mennes et al. (2013), Parr (2007), Sheather (2009), Steen (2011) and Watson (2015).

Files: [query_102.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/query_102.csv), [query_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/query_records.csv), [query_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/query_console.txt).

In [ ]:
#| eval: false
# The query against all 102 sources, and every record it returns. Writes query_102.csv and query_records.csv.
def query(db):
    return f"(({share_near_data(db, 'anywhere', 5)} OR {field(db, 'anywhere', ['\"open data\"', '\"research data management\"'])}) AND {rs(db)})"


rows = [dict(ref_id=s["ref_id"], label=s["label"], year=s["year"], doi=s["doi"], pmid=s["pmid"]) for s in SRC]
for db in DATABASES:
    slug, t = DATABASES[db], [s for s in SRC if sl.testable(db, s)]
    idx = {s["ref_id"] for s in t if sl.count(db, sl.id_clause(db, [s])) > 0}
    fnd = sl.retrieved(db, query(db), t)
    for r in rows:
        r[f"{slug}_indexed"] = "yes" if r["ref_id"] in idx else "no"
        r[f"{slug}_found"] = "yes" if r["ref_id"] in fnd else "no"
sl.write("query_102.csv", rows)
recs = []
for db in DATABASES:
    recs += [dict(r, database=db) for r in fetch(db, query(db), limit_near(db))]
    sl.write("query_records.csv", recs)
print(len(recs), unique_count(recs))

## Reproducibility

We call the database APIs from the analysis server, with the scripts in `retrieval/` and the runs. The local model runs under llama.cpp on the GPU server. Its GGUF file comes from Hugging Face at a fixed revision with its SHA-256 checked, and we start it with `scripts/launch_rds_llm.sh` (`scripts/README.md`). Files the APIs and the local model return go in `data/raw/search-development/`, and files this notebook builds go in `data/processed/search-development/`. The notebook's first code cells build `reference_sources.csv` from the raw files, and they run at every render. The cells of runs 2 and 3 write `terms_counts.csv` and `factor_term_counts.csv` to the same folder. Earlier runs remain in the supplement at the end of the notebook's source, and rendering with `-M exploratory:true` shows them.

The local model runs at temperature 0 with a fixed seed and thinking off, and each output file records the model file, the llama.cpp build and the settings. The databases keep indexing new records, so a rerun of the searches months later returns somewhat different counts; the committed files and `log.md` record what each search returned and when.

The searches need an NCBI API key, a Scopus API key and a Web of Science Starter API key. The Scopus key has full Scopus Search access only from McGill's network, and rebuilding the reference set also needs Elsevier full-text entitlement for the journal.

### Commands

1. **Keys file.** Prompts for each key without echoing it and writes `~/.rds-review-keys` outside the repo, readable by you only.

   ```bash
   umask 077
   read -rsp "NCBI API key: " NCBI; echo
   read -rsp "Scopus API key: " SCOPUS; echo
   read -rsp "Web of Science Starter API key: " WOS; echo
   printf 'export NCBI_API_KEY=%s\nexport SCOPUS_API_KEY=%s\nexport WOS_API_KEY=%s\n' "$NCBI" "$SCOPUS" "$WOS" > ~/.rds-review-keys
   chmod 600 ~/.rds-review-keys
   unset NCBI SCOPUS WOS
   grep -q rds-review-keys ~/.bashrc || echo 'source ~/.rds-review-keys' >> ~/.bashrc
   ls -l ~/.rds-review-keys
   ```

2. **Python environment.**

   ```bash
   cd ~/RDS-Review
   git pull
   python3 -m venv ~/.venvs/rds-review
   source ~/.venvs/rds-review/bin/activate
   pip install --upgrade pip
   pip install -r requirements.txt
   ```

3. **Each script.** Every script runs from the repo root. Paste the block below, with its last line swapped for the step's command from the table.

   ```bash
   cd ~/RDS-Review
   git pull
   source ~/.venvs/rds-review/bin/activate
   source ~/.rds-review-keys
   R=analysis/00_search-development/retrieval
   O=data/raw/search-development
   python3 $R/api_test.py 2>&1 | tee $O/api_test_console.txt
   ```

   In order:

   | Step | Command |
   | --- | --- |
   | API test | `python3 $R/api_test.py 2>&1 \| tee $O/api_test_console.txt` |
   | Reference set | `python3 $R/reference_set.py 2>&1 \| tee $O/reference_set_console.txt` |
   | Lookups | `python3 $R/lookups.py 2>&1 \| tee $O/lookups_console.txt` |
   | Terms the sources use, after the local model is running | `LLM_URL=http://<GPU server>:11451 python3 analysis/00_search-development/local-model/terms.py terms_qwen3.8-27b.csv 2>&1 \| tee $O/terms_console_qwen3.8-27b.txt` |

   Notebook 00 runs after the lookups, to build the processed files. The runs that query the databases are the code cells marked `eval: false` under Open data and research data management. To repeat one, start Python from the repo root with the keys loaded, run the notebook's first code cell and the setup cell under Open data and research data management, then the run's cell.

::: {.content-visible when-meta="exploratory"}

## Supplement

The supplement records the work that came before the query in the main part, from 16 to 25 November 2026. That work tested queries against the known sources as coded by a local model and split by date into development and test sets. The main part tests the query against all 102 sources, with no coding and no split. Rendering with `-M exploratory:true` shows the supplement.


### The coded reference set

We coded each known source against the review's criteria with a local model, and the codes and publication years set the development and test sets. The sections below give the coding, the two sets, and a comparison of Chawinga and Zinn's search statements with the sources they included.


In [ ]:
#| output: false
# The coded reference set, for the exploratory runs.
import re

PROCESSED_FILES |= {"reference_sources_coded.csv", "counts.csv", "verdict.csv", "statement_matches.csv",
                    "factor_theme_agreement.csv"}
CODES = "reference_codes_qwen3.5-122b.csv"
THEMES = ["factors that obstruct", "factors that facilitate", "stakeholder perspectives",
          "policies and what drive them", "mechanisms for implementation"]
STATEMENTS = [("research data sharing", "researchers"), ("digital curation", "data sharing"),
              ("research data sharing", "open science"), ("research data sharing", "open data")]

cand = read("reference_candidates.csv")
look = read("reference_lookups.csv")
codes = read(CODES)


def write(df, name):
    PROCESSED.mkdir(parents=True, exist_ok=True)
    df.to_csv(PROCESSED / name, index=False, lineterminator="\n")


# one value per source for the single-value fields
single = (codes[codes["field"].isin(["practice", "data_origin", "field", "source_type", "language"])]
          .pivot(index="ref_id", columns="field", values="value"))
themes_of = codes[(codes["field"] == "factors") & (codes["value"] != "none")].groupby("ref_id")["theme"].agg(set)

src = look[["ref_id", "title", "pub_date", "doi", "pmid", "abstract_source"]].merge(
    cand[["ref_id", "label", "cited_in"]], on="ref_id").join(single, on="ref_id")
src["year"] = src["pub_date"].str[:4]
src["set"] = src["year"].astype(int).map(lambda y: "development" if y <= 2014 else "test")
src["tier"] = src["field"].map(lambda f: "1" if f == "health" else "2")
src["coded_on"] = src["ref_id"].map(codes.groupby("ref_id")["coded_on"].first())
src["themes"] = src["ref_id"].map(lambda r: "; ".join(sorted(themes_of.get(r, set()))))


def reasons(s, scope):
    """The codebook.yml exclusion codes a source meets, in the codebook's order. Year and tier count
    only for In scope, not for Eligible apart from year."""
    out = []
    if s["practice"] in ("neither", "reuse"):  # the review asks about sharing; reuse alone falls outside it
        out.append("not-data-sharing")
    if not s["themes"]:
        out.append("not-factors")
    if s["data_origin"] == "not-research-data":
        out.append("not-research-data")
    if s["language"] not in ("English", "French"):
        out.append("language")
    if scope and not 2015 <= int(s["year"]) <= 2025:
        out.append("date")
    return out


src["all_reasons"] = src.apply(lambda s: ";".join(reasons(s, True)), axis=1)
src["reason"] = src["all_reasons"].str.split(";").str[0]
src["in_scope"] = src["all_reasons"].map(lambda r: "no" if r else "yes")
src["eligible_apart_from_year"] = src.apply(lambda s: "no" if reasons(s, False) else "yes", axis=1)
src = src[["ref_id", "label", "title", "year", "pub_date", "doi", "pmid", "cited_in", "set", "field", "tier",
           "source_type", "practice", "data_origin", "themes", "language", "coded_on", "abstract_source",
           "eligible_apart_from_year", "in_scope", "reason", "all_reasons"]].sort_values("ref_id")
write(src, "reference_sources_coded.csv")

# counts: sources with each value of each coded field, by set, and codes resting on no passage
setof = src.set_index("ref_id")["set"]
c = codes.assign(set=codes["ref_id"].map(setof), nopassage=codes["passage"].eq("none") & codes["field"].ne("language"))
rows = []
for (field, value, theme), g in c.groupby(["field", "value", "theme"]):
    per = g.groupby("ref_id").agg(set=("set", "first"), nopassage=("nopassage", "all"))
    rows.append(dict(field=field, value=value, theme=theme,
                     development=int((per["set"] == "development").sum()), test=int((per["set"] == "test").sum()),
                     total=len(per), codes_without_passage=int(g["nopassage"].sum())))
for field in ("tier", "eligible_apart_from_year", "in_scope", "reason"):
    for value, g in src.groupby(field):
        rows.append(dict(field=field, value=value or "(none)", theme="",
                         development=int((g["set"] == "development").sum()), test=int((g["set"] == "test").sum()),
                         total=len(g), codes_without_passage=""))
write(pd.DataFrame(rows), "counts.csv")

# verdict: sources eligible apart from year coded for each PCC part, by set
elig = src[src["eligible_apart_from_year"] == "yes"]
has = {"sharing": elig["practice"].isin(["sharing", "both"]), "reuse": elig["practice"].isin(["reuse", "both"])}
has.update({t: elig["themes"].str.contains(t, regex=False) for t in THEMES})
vrows = []
for part, mask in has.items():
    d, t = int((mask & (elig["set"] == "development")).sum()), int((mask & (elig["set"] == "test")).sum())
    verdict = "covered" if d and t else "development only" if d else "test only" if t else "not covered"
    vrows.append(dict(pcc_part=part, development=d, test=t, verdict=verdict))
write(pd.DataFrame(vrows), "verdict.csv")

# Chawinga and Zinn's search statements against titles and abstracts, as phrases and as separate words
squash = lambda s: " " + re.sub(r"[^a-z0-9]+", " ", s.lower()).strip() + " "
text = look.set_index("ref_id").apply(lambda r: squash(r["title"] + " " + r["abstract"]), axis=1)
inscope = set(src.loc[src["in_scope"] == "yes", "ref_id"])
srows = []
for a, b in STATEMENTS:
    for reading in ("phrase", "words"):
        if reading == "phrase":
            hit = lambda t: all(squash(x) in t for x in (a, b))
        else:
            hit = lambda t: all(f" {w} " in t for x in (a, b) for w in squash(x).split())
        ids = sorted(r for r, t in text.items() if hit(t))
        srows.append(dict(statement=f"{a} AND {b}", reading=reading, matched=len(ids),
                          matched_in_scope=sum(r in inscope for r in ids), ref_ids=";".join(ids)))
write(pd.DataFrame(srows), "statement_matches.csv")
print(f"Built reference_sources_coded.csv ({len(src)} sources), counts.csv, verdict.csv and statement_matches.csv.")

steps_1_2 = ["reference_candidates.csv", "reference_sources_coded.csv", CODES, "counts.csv",
             "verdict.csv", "statement_matches.csv"]
missing = [n for n in steps_1_2 if not path(n).exists()]
have_data = not missing
if not have_data:
    print("Not run yet. Missing: " + ", ".join(missing))
else:
    cand, src, codes, counts, verdict, statements = (read(n) for n in steps_1_2)


#### Coding

A local model, Qwen3.5-122B-A10B, coded each source from its title and abstract with `prompts/coding_prompt.md`. The prompt holds the definitions. The model recorded whether the source concerns sharing, reuse, both or neither, whether the data are research data, the factors the source reports under the screening guide's five themes, its field, its source type and its language, each with the sentence that supports it. A smaller model, Qwen3.8-27B, also coded all 102 sources with `coding_prompt.md`, and the two agree on 79% to 100% of sources for each variable.

Most sources concern sharing: 63 are coded `sharing` and 22 `both`. Policies (47), infrastructure (46) and barriers (31) are the factors most reported. 14 sources report no factor on the list. The codes that rest on no passage are mostly factors `none` (14), `data_origin` `unclear` (10) and `source_type` `other` (7). Each marks a detail the title and abstract leave out. Four codes carry a problem from the coding checks. One passage is not in the title or abstract, and three carry a copied "Title:" label.

We also check that every coded source is in the reference set and that each source has exactly one code for each variable that takes one value. All checks pass.

Files: [counts.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/processed/search-development/counts.csv), [reference_codes_qwen3.5-122b.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/reference_codes_qwen3.5-122b.csv).

In [4]:
#| output: false
if have_data:
    errors = []
    ids = set(src["ref_id"])
    if set(cand["ref_id"]) != ids:
        errors.append(f"reference_sources_coded.csv and reference_candidates.csv list different sources: {sorted(set(cand['ref_id']) ^ ids)}.")
    unknown = set(codes["ref_id"]) - ids
    if unknown:
        errors.append(f"{CODES} has sources not in reference_sources_coded.csv: {sorted(unknown)}.")
    for f in ("practice", "data_origin", "field", "source_type", "language"):
        n = codes[codes["field"] == f].groupby("ref_id").size().reindex(sorted(ids), fill_value=0)
        if (n != 1).any():
            errors.append(f"{int((n != 1).sum())} source(s) lack exactly one {f} code: {sorted(n[n != 1].index)}.")
    if errors:
        raise ValueError("\n".join(errors))
    print("All checks passed.")
    flagged = codes[codes["problem"] != ""]
    if len(flagged):
        print(f"{len(flagged)} code(s) in {CODES} carry a problem from the coding checks:")
        for _, f in flagged.iterrows():
            print(f"  {f['ref_id']}, {f['field']} {f['value']}: {f['problem']}")


All checks passed.
4 code(s) in reference_codes_qwen3.5-122b.csv carry a problem from the coding checks:
  bb0315, factors governance: passage not found in the title or abstract
  bb0500, field other: passage not found in the title or abstract
  bb0525, field other: passage not found in the title or abstract
  bb2060, field other: passage not found in the title or abstract


#### Development and test sets

We applied the RDS-Review criteria to the codes. A source is eligible apart from year when it concerns sharing, alone or with reuse, is not coded as concerning data from outside research, reports at least one factor, and is in English or French. Sources published 2007 to 2014 form the development set, and the runs measure candidate queries against it. Sources published from 2015 form the test set. We held it out while choosing the title query, then measured the title query against it (exploratory run 19). Exploratory runs 20 to 25 count test sources, so the test set did not stay out of choosing the revised query.

45 of the 60 development sources and 31 of the 42 test sources are eligible apart from year, and 39 and 28 of them have a DOI or PMID, so we can check the databases for them. Of the sources that are not eligible, most fall under `not-data-sharing` (8 development, 9 test), the rest under `not-factors` (5 and 1) and `not-research-data` (2 and 1). Every development source fails the date criterion, since the set ends in 2014.

A PCC part is covered when sources eligible apart from year carry it in both sets. All seven PCC parts are covered in both sets. Stakeholder perspectives rests on 6 development and 4 test sources, the fewest of any part.

Files: [reference_sources.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/processed/search-development/reference_sources.csv), [verdict.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/processed/search-development/verdict.csv).

#### How their search compares

Chawinga and Zinn's criteria are narrower on source type, empirical design and language, and overlap with RDS-Review's on topic and date. They set no criterion on reuse, data origin, factors or field, so we used the codes to see what their set covers.

We matched the four search statements Chawinga and Zinn report against the titles and abstracts of the reference set, once with each part as a phrase and once as separate words. Read as phrases, the four statements match at most 2 sources; read as separate words, at most 29, 11 of them in scope. Chawinga and Zinn's statements match a minority of the sources they included, so we did not model the RDS-Review query on them.

Files: [comparison.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/comparison.csv), [statement_matches.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/processed/search-development/statement_matches.csv).


### Exploratory runs

Exploratory runs 3 to 31 are numbered in the order we ran them. Each phase ends in a decision that set up the next. The last phase, open near data, led to the phrases "open data" and "research data management" in runs 4 and 5 of the main part.

Each run measures candidate queries in PubMed, Scopus and Web of Science by the records they return for 2015 to 2025 in English or French, and by the development sources they find, checked by DOI or PMID with no date limit. A candidate drops out when another finds at least as many development sources in fewer records. The test set stays out of exploratory runs 5 to 18. We ran exploratory runs 5 to 16 on 18 November 2026. They count development sources out of 40 with a DOI or PMID, because they ran before we removed sources about reuse alone from the known sources; the development set now has 39.

The runs query the databases' APIs, so they run on the analysis server from the repository root, with the keys loaded (`source ~/.rds-review-keys`). Their cells carry `eval: false` and do not run at render. To repeat a run, run the setup cell and then the run's cell.

In [ ]:
#| eval: false
# Setup for the exploratory runs, after the setup cell under Open data and research data management.
# The sharing block that exploratory run 5 settled on.
SHARING = {"PubMed": ['"data sharing"', '"shar* data"'],
           "Scopus": ['"data sharing"'],
           "Web of Science": ['"data sharing"']}

# The candidate factor terms (listed under Phrase and factor terms), then general words for a factor that the list lacks.
CANDIDATE_TERMS = ["polic*", "challenge*", "repositor*", "ethic*", "privacy", "infrastructure", "barrier*",
                   "perception*", '"open science"', "motivat*", "interoperab*", '"open data"', '"open access"',
                   "consent", '"open research"', "licens*", "attitude*", "incentive*", "mandate*", "willingness",
                   "obstacle*", "stewardship", '"data management plan*"', '"data curation"', "facilitator*",
                   "enabler*", "reluctan*", '"digital curation"']
GENERAL_WORDS = ["factor*", "influenc*", "determinant*", "driver*", "concern*", "trust"]
ALL_FACTORS = CANDIDATE_TERMS + GENERAL_WORDS
ZERO_STUDY = ["facilitator*", "driver*", "interoperab*", '"open research"', '"digital curation"']  # exploratory run 9
OUT_OF_SCOPE = ["privacy"]  # exploratory run 10
FACTORS = [t for t in ALL_FACTORS if t not in ZERO_STUDY + OUT_OF_SCOPE]


def version3(db, factors, near):
    """Data sharing in the title, OR data sharing with a factor term anywhere, or within 5 words with near."""
    v1 = field(db, "title", SHARING[db])
    s, f = " OR ".join(SHARING[db]), " OR ".join(factors)
    if not near:
        v2 = f"({field(db, 'anywhere', SHARING[db])} AND {field(db, 'anywhere', factors)})"
    elif db == "Scopus":
        v2 = f"TITLE-ABS-KEY(({s}) W/5 ({f}))"
    else:
        v2 = f"TS=(({s}) NEAR/5 ({f}))"
    return f"({v1} OR {v2})"


CODED = sl.read("reference_sources_coded.csv")
DEV = [s for s in CODED if s["set"] == "development" and s["eligible_apart_from_year"] == "yes"]
LABEL = {s["ref_id"]: s["label"] for s in CODED}


def dev_in(db):
    """Development studies a database can be checked for: a DOI, or in PubMed a PMID."""
    return [s for s in DEV if sl.testable(db, s)]


#### Terms and codes

In exploratory runs 3 and 4, we compared the local model's factor themes and the terms it extracted with the factor term match of run 3. The three reach similar numbers of sources, but the extracted terms name sharing and the other two name factors.

##### Exploratory run 3. Factor themes coded by Qwen against the factor term match

**Question.** Does the local model's coding of factor themes differ from a match of the factor terms?

**Measured.** For each of the five themes, and for any theme, each source gets a yes or no from the factor themes Qwen3.5-122B coded (the themes in `reference_sources.csv`) and from run 3's factor term match. We report percentage agreement, Cohen's kappa with a 95% confidence interval, and McNemar's exact test, for all 102 sources and for the development and test sets.

**Result.** Factor themes coded by Qwen and factor term match agree on 63% to 90% of sources per theme, with kappa from 0.34 to 0.64. Qwen codes more themes than factor term match finds. For factors that obstruct and factors that facilitate, factor term match finds no theme that Qwen misses, while Qwen codes factors that obstruct in 19 sources and factors that facilitate in 38 sources with no factor term for them (McNemar p < 0.001 for both). For policies and what drive them, Qwen alone codes 14 sources and factor term match alone finds 4 (p = 0.031). For stakeholder perspectives and mechanisms for implementation, the disagreements run both ways and neither direction prevails (McNemar p = 0.34 and 0.33). For any theme, Qwen codes a factor in 88 sources and factor term match finds one in 79; the two agree on 81 sources (79%), with kappa 0.32.

**Decision.** Qwen finds factors in sources that use none of the factor terms. Factors stay out of the query and go to screening (exploratory run 15), so we take no terms from the coding.

Files: [factor_theme_agreement.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/processed/search-development/factor_theme_agreement.csv), [reference_sources.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/processed/search-development/reference_sources.csv), [reference_codes_qwen3.5-122b.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/reference_codes_qwen3.5-122b.csv).

In [7]:
def agreement(a, b, a_name, b_name):
    """Percentage agreement, Cohen's kappa with a 95% confidence interval, and McNemar's exact test
    for two yes-or-no codings of the same sources."""
    n = len(a)
    yy, yn, ny, nn = int((a & b).sum()), int((a & ~b).sum()), int((~a & b).sum()), int((~a & ~b).sum())
    po = (yy + nn) / n
    pe = ((yy + yn) * (yy + ny) + (ny + nn) * (yn + nn)) / n ** 2
    kappa = (po - pe) / (1 - pe) if pe < 1 else float("nan")
    se = math.sqrt(po * (1 - po) / (n * (1 - pe) ** 2)) if pe < 1 else float("nan")
    d = yn + ny
    p = min(1.0, 2 * sum(math.comb(d, k) for k in range(min(yn, ny) + 1)) / 2 ** d) if d else 1.0
    return {"n": n, "both": yy, f"{a_name} only": yn, f"{b_name} only": ny, "neither": nn, **dict( agreement=round(po, 2),
                kappa=round(kappa, 2), kappa_low=round(max(-1.0, kappa - 1.96 * se), 2), kappa_high=round(min(1.0, kappa + 1.96 * se), 2),
                mcnemar_p=round(p, 3))}


if path("reference_lookups.csv").exists():
    sources = read("reference_sources_coded.csv").set_index("ref_id")
    sets = sources["set"].reindex(themes.index)
    coded = sources["themes"].reindex(themes.index).map(lambda s: {t.strip() for t in s.split(";") if t.strip()})
    qwen = pd.DataFrame({theme: coded.map(lambda s, t=theme: t in s) for theme in TABLE1})
    qwen["any theme"] = qwen.any(axis=1)
    keywords = themes[list(TABLE1)].copy()
    keywords["any theme"] = keywords.any(axis=1)
    compared = pd.DataFrame([dict(theme=t, set=name, **agreement(qwen.loc[rows, t], keywords.loc[rows, t], "Factor themes coded by Qwen", "Factor term match"))
                             for t in qwen.columns
                             for name, rows in (("all", qwen.index), ("development", sets == "development"),
                                                ("test", sets == "test"))]).set_index(["theme", "set"])
    write(compared.reset_index(), "factor_theme_agreement.csv")
    display(compared.xs("all", level="set")[["agreement", "kappa", "kappa_low", "kappa_high", "mcnemar_p"]])
else:
    print("Not run yet: reference_lookups.csv is missing.")

,agreement,kappa,kappa_low,kappa_high,mcnemar_p
theme,,,,,
factors that obstruct,0.81,0.63,0.48,0.78,0.000
factors that facilitate,0.63,0.34,0.18,0.51,0.000
stakeholder perspectives,0.90,0.53,0.25,0.81,0.344
policies and what drive them,0.82,0.64,0.49,0.79,0.031
mechanisms for implementation,0.75,0.46,0.28,0.64,0.327
any theme,0.79,0.32,0.05,0.58,0.078


##### Exploratory run 4. Terms extracted by Qwen, factor term match and factor themes coded by Qwen

**Question.** Do the three approaches reach the same sources?

- **Terms extracted by Qwen:** run 2, in which Qwen3.8-27B listed the words each source uses to describe research data sharing.
- **Factor term match:** run 3, in which we check each factor term in the screening guide against each source's title and abstract, with no model.
- **Factor themes coded by Qwen:** the coding of the known sources, in which Qwen3.5-122B judged which of the screening guide's five factor themes each source reports.

**Measured.** Whether each approach reaches a source: a term extracted by Qwen, a factor term, or any factor theme coded by Qwen. For the three together we report Fleiss' kappa and Cochran's Q, and for each pair percentage agreement, Cohen's kappa and McNemar's exact test.

**Result.** Terms extracted by Qwen reach 84 sources, factor term match 79 and factor themes coded by Qwen 88. All three reach 67 sources and none reaches 4; the other 31 split. Fleiss' kappa for the three is 0.31, and Cochran's Q is 3.9 (p = 0.14), so the three do not differ significantly in how many sources each reaches. Pairwise, kappa is 0.48 for terms extracted by Qwen and factor themes coded by Qwen, 0.32 for factor term match and factor themes coded by Qwen, and 0.18 for terms extracted by Qwen and factor term match. Terms extracted by Qwen name sharing, and the other two name factors.

Files: [reference_sources.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/processed/search-development/reference_sources.csv), [reference_codes_qwen3.5-122b.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/reference_codes_qwen3.5-122b.csv), [terms_qwen3.8-27b.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/terms_qwen3.8-27b.csv).

In [8]:
def fleiss_kappa(frame):
    """Fleiss' kappa for several yes-or-no codings of the same sources."""
    k, n = frame.shape[1], len(frame)
    yes = frame.sum(axis=1)
    p_yes = yes.sum() / (n * k)
    p_i = (yes * (yes - 1) + (k - yes) * (k - yes - 1)) / (k * (k - 1))
    pe = p_yes ** 2 + (1 - p_yes) ** 2
    return (p_i.mean() - pe) / (1 - pe)


def cochran_q(frame):
    """Cochran's Q for several yes-or-no codings of the same sources, with its p-value."""
    k = frame.shape[1]
    cols, rows = frame.sum(axis=0), frame.sum(axis=1)
    q = (k - 1) * (k * (cols ** 2).sum() - cols.sum() ** 2) / (k * rows.sum() - (rows ** 2).sum())
    # chi-squared with k - 1 degrees of freedom; for k = 3 the survival function is exp(-q / 2)
    return q, math.exp(-q / 2) if k == 3 else float("nan")


if path("reference_lookups.csv").exists() and path("terms_qwen3.8-27b.csv").exists():
    term_rows = read("terms_qwen3.8-27b.csv")
    with_terms = set(term_rows.loc[term_rows["location"] != "not in text", "ref_id"])
    three = pd.DataFrame({"Terms extracted by Qwen": qwen.index.isin(with_terms),
                          "Factor term match": keywords["any theme"].to_numpy(),
                          "Factor themes coded by Qwen": qwen["any theme"].to_numpy()}, index=qwen.index)
    patterns = three.value_counts().rename("sources").reset_index()
    pairs = [pd.DataFrame([agreement(three[a], three[b], a, b)], index=[f"{a} and {b}"])
             for a, b in (('Terms extracted by Qwen', 'Factor term match'), ('Terms extracted by Qwen', 'Factor themes coded by Qwen'), ('Factor term match', 'Factor themes coded by Qwen'))]
    q, p = cochran_q(three.astype(int))
    print(f"sources each approach reaches: " + ", ".join(f"{c} {int(three[c].sum())}" for c in three))
    print(f"Fleiss' kappa {fleiss_kappa(three.astype(int)):.2f}; Cochran's Q {q:.1f}, p = {p:.3f}")
    display(pd.concat(pairs)[["agreement", "kappa", "mcnemar_p"]])

sources each approach reaches: Terms extracted by Qwen 84, Factor term match 79, Factor themes coded by Qwen 88
Fleiss' kappa 0.31; Cochran's Q 3.9, p = 0.140


,agreement,kappa,mcnemar_p
Terms extracted by Qwen and Factor term match,0.74,0.18,0.442
Terms extracted by Qwen and Factor themes coded by Qwen,0.86,0.48,0.424
Factor term match and Factor themes coded by Qwen,0.79,0.32,0.078


#### Phrase and factor terms

The phrase "data sharing", alone or with factor terms, missed sources that word sharing differently (exploratory run 13), which led to share near data. Exploratory runs 5 to 13 test a phrase for data sharing, alone and combined with factor terms. We drafted the candidate factor terms from the PCC table's factor themes before search development began: policies, policy, challenge\*, repositor\*, ethic\*, privacy, infrastructure, barrier\*, perception\*, open science, motivat\*, interoperab\*, "open data", open access, consent, open research, licens\*, attitude\*, incentive\*, mandate\*, willingness, obstacle\*, stewardship, "data management plan\*", "data curation", facilitator\*, enabler\*, reluctan\* and "digital curation". "data reuse" and "data re-use" were the candidates for reuse. We later dropped reuse from the review question.

##### Exploratory run 5. Sharing terms

**Question.** Does any phrase for sharing data find development sources that *"data sharing"* misses, at a cost worth paying?

**Measured.** 18 candidate phrases for sharing, among them *"shar\* data"*, *"data access"*, *"data availability"* and *"data publication"*, each alone in title, abstract and keywords in the three databases. For each, the records it returns, the development sources it finds, how many of them *"data sharing"* misses, and how many no other phrase finds.

**Result.** In PubMed, *"shar\* data"* finds three development sources *"data sharing"* misses: Bertzky and Stoll-Kleemann (2009), Cooper (2007), and Rohlfing and Poline (2012). *"data publication"*, *"data access"* and *"data availability"* each add only Costello (2009), for 600 to 18,000 records.

**Decision.** The phrase becomes *"data sharing"*, plus *"shar\* data"* in PubMed. The other phrases stay out.

Files: [term_table_sharing.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_sharing.csv).

In [10]:
#| eval: false
SHARING_CANDIDATES = ['"data sharing"', '"research data sharing"', '"shar* data"', '"shared data"',
                      '"sharing of data"', '"open data sharing"', '"data access"', '"data availability"',
                      '"data publication"', '"data dissemination"', '"data deposit*"', '"data repositor*"',
                      '"open data"', '"data reuse"', '"data re-use"', '"secondary use"', '"FAIR data"',
                      '"FAIR principles"']
MESH = '"Information Dissemination"[mh:noexp]'  # PubMed only

rows = []
for db in DATABASES:
    dev = dev_in(db)
    terms = SHARING_CANDIDATES + ([MESH] if db == "PubMed" else [])
    found = {}
    for t in terms:
        w = t if t == MESH else TAG["anywhere"][db].format(t)
        found[t] = (sl.count(db, f"({w}) AND {limit(db)}"), sl.retrieved(db, w, dev))
    for t in terms:
        n, hits = found[t]
        others = set().union(*(found[o][1] for o in terms if o != t))
        rows.append(dict(database=db, term="Information Dissemination [MeSH]" if t == MESH else t,
                         records_2015_2025=n, development_found=len(hits),
                         beyond_data_sharing=len(hits - found['"data sharing"'][1]),
                         only_this_term=len(hits - others), limit=limit(db),
                         studies="; ".join(sorted(LABEL[r] for r in hits))))
sl.write("term_table_sharing.csv", rows)

##### Exploratory runs 6 and 7. Small blocks of factor terms

**Question.** Which few factor terms, paired with the phrase, find the most development sources for the fewest records?

**Measured.** Exploratory run 6 pairs each candidate factor term with the phrase, anywhere in the record and, in Scopus and Web of Science, within 5 words of it, then builds a block one term at a time, each time adding the term with the most new sources per extra record. Exploratory run 7 tries every combination of up to seven terms offline, from the sources each term finds.

**Result.** The best blocks have five to seven terms. Anywhere in the record, they find 18 sources in PubMed for 3,452 records, 24 in Scopus for 6,805 and 26 in Web of Science for 8,656.

**Decision.** We could not justify five to seven terms chosen by yield over the rest of the list. Exploratory runs 9 to 12 test the whole list of candidate terms in their place, with six general words for a factor that Claude proposed: *factor\**, *influenc\**, *determinant\**, *driver\**, *concern\** and *trust*.

Files: [term_table_factors.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_factors.csv), [term_table_factor_combinations.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_factor_combinations.csv).

In [11]:
#| eval: false
def sharing_and(db, factors, near=False):
    """The sharing block AND any factor term: anywhere in the record, or within 5 words."""
    s, f = " OR ".join(SHARING[db]), " OR ".join(factors)
    if db == "PubMed":
        return field(db, "anywhere", SHARING[db]) + (f" AND {field(db, 'anywhere', factors)}" if factors else "")
    if not factors:
        return field(db, "anywhere", SHARING[db])
    if near:
        return f"TITLE-ABS-KEY(({s}) W/5 ({f}))" if db == "Scopus" else f"TS=(({s}) NEAR/5 ({f}))"
    return f"{field(db, 'anywhere', SHARING[db])} AND {field(db, 'anywhere', factors)}"


# Exploratory run 6
rows = []
for db in DATABASES:
    dev = dev_in(db)
    base = sl.retrieved(db, sharing_and(db, []), dev)
    for near in ([False] if db == "PubMed" else [False, True]):  # PubMed has no proximity for truncated terms
        mode = "within 5 words" if near else "anywhere in title or abstract"
        found = {f: (sl.count(db, f"({sharing_and(db, [f], near)}) AND {limit(db)}"),
                     sl.retrieved(db, sharing_and(db, [f], near), dev)) for f in ALL_FACTORS}
        for f in ALL_FACTORS:
            rows.append(dict(database=db, placement=mode, step="", term=f, records_2015_2025=found[f][0],
                             development_found=len(found[f][1]), of=len(base),
                             studies="; ".join(sorted(LABEL[r] for r in found[f][1]))))
        # the block one term at a time: most new studies per extra record first
        chosen, got, load = [], set(), 0
        while True:
            best = None
            for f in ALL_FACTORS:
                new = found[f][1] - got
                if f in chosen or not new:
                    continue
                n = sl.count(db, f"({sharing_and(db, chosen + [f], near)}) AND {limit(db)}")
                if best is None or len(new) / max(n - load, 1) > best[0]:
                    best = (len(new) / max(n - load, 1), f, n, new)
            if best is None:
                break
            _, f, load, new = best
            chosen.append(f)
            got |= new
            rows.append(dict(database=db, placement=mode, step=len(chosen), term=f, records_2015_2025=load,
                             development_found=len(got), of=len(base),
                             studies="; ".join(sorted(LABEL[r] for r in got))))
sl.write("term_table_factors.csv", rows)

# Exploratory run 7: a combination's studies are the union of its terms' studies, so they are worked out offline; only the
# record counts of the best combinations go to the databases.
MAX_TERMS, VERIFY = 7, 3
table = [r for r in sl.read("term_table_factors.csv") if r["step"] == ""]
rows = []
for (db, mode), group in itertools.groupby(table, key=lambda r: (r["database"], r["placement"])):
    group = [r for r in group if int(r["development_found"]) > 0]
    near = mode == "within 5 words"
    studies = sorted({s for r in group for s in r["studies"].split("; ")})
    bit = {s: 1 << i for i, s in enumerate(studies)}
    terms = [(r["term"], int(r["records_2015_2025"]), sum(bit[s] for s in r["studies"].split("; ")))
             for r in group]
    best = {}
    for k in range(1, min(MAX_TERMS, len(terms)) + 1):
        for combo in itertools.combinations(terms, k):
            mask = 0
            for _, _, m in combo:
                mask |= m
            # records estimated as the sum of the terms' own records, to rank; the real count comes next
            best.setdefault(bin(mask).count("1"), []).append((sum(c[1] for c in combo), [c[0] for c in combo]))
    for n in sorted(best, reverse=True):
        for _, combo in sorted(best[n])[:VERIFY]:
            rows.append(dict(database=db, placement=mode, studies_kept=n,
                             records_2015_2025=sl.count(db, f"({sharing_and(db, combo, near)}) AND {limit(db)}"),
                             terms=" OR ".join(combo)))
sl.write("term_table_factor_combinations.csv", rows)

##### Exploratory run 8. The phrase in the title, or anywhere

**Question.** Where must the phrase appear? Every factor term combines with it, so we settled its placement before choosing factor terms.

**Measured.** Records and development sources found for the phrase in the title alone, and anywhere in title, abstract and keywords.

**Result.** The title alone cuts records by about 80%, and loses 4 development sources in PubMed, 12 in Scopus and 13 in Web of Science.

**Decision.** The phrase alone in the title loses too many sources. We tested factor terms with the phrase anywhere in the record (exploratory run 9).

Files: [term_table_sharing_field.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_sharing_field.csv).

In [12]:
#| eval: false
rows = []
for db in DATABASES:
    for where in ("title", "anywhere"):
        q = field(db, where, SHARING[db])
        hits = sl.retrieved(db, q, dev_in(db))
        rows.append(dict(database=db, field=where, query=q, limit=limit(db),
                         records_2015_2025=sl.count(db, f"{q} AND {limit(db)}"), development_found=len(hits),
                         studies="; ".join(sorted(LABEL[r] for r in hits))))
sl.write("term_table_sharing_field.csv", rows)

##### Exploratory run 9. Phrase plus factor words

**Question.** Does adding factor terms to the phrase find more development sources, and at what cost?

**Measured.** Three versions, each with the whole block of candidate factor terms:

1. the phrase in the title, with no factor terms;
2. the phrase anywhere AND a factor term anywhere;
3. version 1 OR version 2, which later runs call phrase plus factor words.

Versions 2 and 3 each run once with the full block and once without the five terms that find no development source in any database: *facilitator\**, *driver\**, *interoperab\**, *"open research"* and *"digital curation"*.

**Result.** Records for 2015 to 2025, and development sources found, without those five terms:

| Version | PubMed | Scopus | Web of Science |
|:--|--:|--:|--:|
| 1. The phrase, title | 1,677 and 17 | 5,517 and 16 | 4,438 and 16 |
| 2. The phrase and a factor term, anywhere | 6,959 and 18 | 22,134 and 24 | 15,057 and 26 |
| 3. Version 1 OR version 2 | 7,525 and 21 | 23,357 and 25 | 16,208 and 27 |

The five terms add records and no sources in every database.

**Decision.** The five terms go. All three versions survive the rule, each finding more sources than the one before at more records.

Files: [term_table_structure.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_structure.csv).

In [13]:
#| eval: false
blocks = {"full": ALL_FACTORS, "full minus zero-study terms": [t for t in ALL_FACTORS if t not in ZERO_STUDY]}
rows = []
for db in DATABASES:
    dev = dev_in(db)
    res = {"1 title only": field(db, "title", SHARING[db])}
    for b, factors in blocks.items():
        res[f"2 anywhere AND factor, {b}"] = (f"({field(db, 'anywhere', SHARING[db])} AND "
                                              f"{field(db, 'anywhere', factors)})")
        res[f"3 version 1 OR version 2, {b}"] = version3(db, factors, near=False)
    res = {k: (q, sl.count(db, f"{q} AND {limit(db)}"), sl.retrieved(db, q, dev)) for k, q in res.items()}
    for k, (q, n, hits) in res.items():
        beaten = [o for o, (_, n2, h2) in res.items() if o != k and len(h2) >= len(hits) and n2 < n]
        rows.append(dict(database=db, version=k, query=q, limit=limit(db), records_2015_2025=n,
                         development_found=len(hits), of=len(dev), beaten_by="; ".join(beaten),
                         studies="; ".join(sorted(LABEL[r] for r in hits))))
sl.write("term_table_structure.csv", rows)

##### Exploratory run 10. What each factor term costs

**Question.** Does any single factor term carry the load, or cost much on its own?

**Measured.** Phrase plus factor words with the 29 remaining terms, removing one term at a time and putting it back. For each term, the records it alone adds and the development sources no other term finds.

**Result.** Only *repositor\** in PubMed (Coady and Wagner 2013) and *challenge\** in Web of Science (Van Horn and Gazzaniga 2013) find a source no other term finds. Removing *privacy* loses no source in any database.

**Decision.** *privacy* leaves the block. Privacy stays in the screening guide as a factor that obstructs sharing.

Files: [term_table_leave_one_out.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_leave_one_out.csv).

In [14]:
#| eval: false
factors = [t for t in ALL_FACTORS if t not in ZERO_STUDY]
rows = []
for db in DATABASES:
    dev = dev_in(db)
    q = version3(db, factors, near=False)
    n0, h0 = sl.count(db, f"{q} AND {limit(db)}"), sl.retrieved(db, q, dev)
    for t in factors:
        q = version3(db, [f for f in factors if f != t], near=False)
        n, h = sl.count(db, f"{q} AND {limit(db)}"), sl.retrieved(db, q, dev)
        lost = sorted(LABEL[r] for r in h0 - h)
        rows.append(dict(database=db, term_removed=t, records_with_all=n0, records_saved=n0 - n,
                         studies_with_all=len(h0), studies_lost=len(lost), lost="; ".join(lost)))
sl.write("term_table_leave_one_out.csv", rows)

##### Exploratory run 11. Removing factor terms one after another, and proximity

**Question.** Can removing factor terms bring the load down without losing sources, and does requiring the factor term near the phrase help?

**Measured.** Phrase plus factor words with 28 terms, removing one term per step without putting it back, in the order of exploratory run 10. Terms that lose no source go first, and among them the terms that save the most records. In Scopus and Web of Science the run repeats with the factor term within 5 words of the phrase.

**Result.** Sources drop out by the second or third step in Scopus and Web of Science. With one factor term left, phrase plus factor words still returns 2,209 records in PubMed, 5,632 in Scopus and 9,051 in Web of Science. The order removes *polic\** and *barrier\** early, because they cost the most records. Proximity halves the load. With all 28 terms, Scopus returns 9,018 records against 21,664, and Web of Science 7,175 against 15,294.

**Decision.** Cutting terms cannot bring the load down far, and the cuts remove *polic\** and *barrier\** early, two terms a query about factors needs. Phrase plus factor words keeps all 28 terms, within 5 words of the phrase in Scopus and Web of Science.

Files: [term_table_cuts.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_cuts.csv).

In [15]:
#| eval: false
loo = sl.read("term_table_leave_one_out.csv")
rows = []
for db in DATABASES:
    dev = dev_in(db)
    order = [r["term_removed"] for r in sorted((r for r in loo if r["database"] == db),
             key=lambda r: (int(r["studies_lost"]), -int(r["records_saved"])))
             if r["term_removed"] not in OUT_OF_SCOPE]
    for near in ([False] if db == "PubMed" else [False, True]):
        kept = list(order)
        h_start = sl.retrieved(db, version3(db, kept, near), dev)
        for step in range(len(order)):
            cut = order[step - 1] if step else ""
            if cut:
                kept.remove(cut)
            q = version3(db, kept, near)
            n, h = sl.count(db, f"{q} AND {limit(db)}"), sl.retrieved(db, q, dev)
            rows.append(dict(database=db, placement="within 5 words" if near else "anywhere", step=step,
                             term_cut=cut, terms_left=len(kept), records_2015_2025=n, development_found=len(h),
                             of=len(dev), lost_so_far="; ".join(sorted(LABEL[r] for r in h_start - h)),
                             query=q, limit=limit(db)))
sl.write("term_table_cuts.csv", rows)

##### Exploratory run 12. Records of phrase plus factor words

**Question.** How many records does phrase plus factor words return across the three databases?

**Measured.** Phrase plus factor words with 28 terms, anywhere in PubMed and within 5 words in Scopus and Web of Science. We counted every record the three return, with and without duplicates. Duplicates here share a DOI, or a title and year. bib-dedupe deduplicates the review's records and finds as many duplicates or more.

**Result.** 23,488 records, or 14,255 without duplicates.

**Decision.** Too many to screen. Exploratory run 15 sets it against the queries that replace it.

Files: [final_query_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/final_query_records.csv).

In [16]:
#| eval: false
for db in ["Scopus", "Web of Science"]:
    lost = sl.retrieved(db, version3(db, FACTORS, False), dev_in(db)) - sl.retrieved(db, version3(db, FACTORS, True), dev_in(db))
    print(f"{db}, found anywhere but not within 5 words: {'; '.join(sorted(LABEL[r] for r in lost)) or '-'}")

recs = []
for db in DATABASES:
    q = f"{version3(db, FACTORS, near=db != 'PubMed')} AND {limit(db)}"
    if db == "PubMed":
        got = list(sl.pubmed_summary(sl.pubmed_all_ids(q)).values())
    elif db == "Scopus":
        got = sl.scopus_records(q)
    else:
        got = sl.wos_records(q)
    recs += [dict(r, database=db) for r in got]
sl.write("final_query_records.csv", recs)

##### Exploratory run 13. Development sources phrase plus factor words misses

**Question.** Which development sources does phrase plus factor words miss, and why?

**Measured.** Each development source with a DOI or PMID, checked in each database. For each source none finds, whether the databases index it.

**Result.** The three databases together find 28 of 40. Five are in none of the databases. The query misses Borgman (2012), Cohn (2012), Costello (2009), Haddow et al. (2011), McCullough (2009), Parr (2007) and Zimmerman (2008), although the databases index them. Three of them word sharing in ways the phrase does not match.

**Decision.** Exploratory run 14 tests a form of *share* near *data* in place of the phrase.

Files: [final_query_missed.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/final_query_missed.csv).

In [17]:
#| eval: false
found, indexed = {}, {}
for db in DATABASES:
    found[db] = sl.retrieved(db, version3(db, FACTORS, near=db != "PubMed"), dev_in(db))
    indexed[db] = {s["ref_id"] for s in dev_in(db) if sl.count(db, sl.id_clause(db, [s])) > 0}
anyf = set().union(*found.values())

rows = []
for s in sorted(DEV, key=lambda s: s["label"]):
    if s["ref_id"] in anyf:
        continue
    where = {db: ("no DOI or PMID" if not (s["doi"] or s["pmid"]) else
                  "not indexed" if s["ref_id"] not in indexed[db] else "indexed, not found") for db in DATABASES}
    rows.append(dict(ref_id=s["ref_id"], label=s["label"], title=s["title"], year=s["year"], doi=s["doi"],
                     pmid=s["pmid"], field=s["field"], source_type=s["source_type"],
                     pubmed=where["PubMed"], scopus=where["Scopus"], web_of_science=where["Web of Science"]))
sl.write("final_query_missed.csv", rows)

#### Share near data

A form of share within five words of data in titles found more development sources in fewer records than the phrase with factor words, and became the title query (exploratory run 15). The search kept every document type (exploratory run 16). The phrase misses development sources whose titles say "sharing research data" or "sharing of health data" (exploratory run 13). Exploratory runs 14 to 16 search for a form of *share* near *data* instead.

##### Exploratory run 14. Share near data, in place of the phrase

**Question.** Exploratory run 13 found three development sources missed because their titles word sharing differently: "sharing research data" (Borgman 2012), "sharing of health data" (Haddow et al. 2011) and "sharing and reuse of ecological data" (Zimmerman 2008). Does share near data find them, and at what cost?

**Measured.** The phrase against share within 3 and within 5 words of *data*, in the title and anywhere in the record. PubMed's proximity search takes no truncation, so there the block writes out *sharing*, *shared*, *share* and *shares*.

**Result.** Records for 2015 to 2025, and development sources found:

| | PubMed | Scopus | Web of Science |
|:--|--:|--:|--:|
| The phrase, title | 1,677 and 17 | 5,517 and 16 | 4,438 and 16 |
| Share within 3 words, title | 2,385 and 20 | 8,725 and 26 | 7,148 and 27 |
| Share within 5 words, title | 2,532 and 20 | 9,547 and 27 | 7,813 and 28 |
| Share within 5 words, anywhere | 18,840 and 21 | 82,908 and 30 | 62,198 and 31 |

In the title, share near data finds 26 to 28 development sources in Scopus and Web of Science with no factor words, where phrase plus factor words finds 21 for a similar number of records. Anywhere, it returns 7 to 9 times the records for one to three more sources. Web of Science's API rejects the language limit beside a proximity search, so its counts here cover all languages; elsewhere the limit removes about 0.5% of records.

**Decision.** Share near data in the title replaces the phrase and the factor words. We left searching anywhere out. Exploratory run 15 compares the 3- and 5-word versions across the three databases together.

Files: [term_table_sharing_proximity.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_sharing_proximity.csv).

In [18]:
#| eval: false
rows = []
for db in DATABASES:
    dev = dev_in(db)
    phrase = {}
    for where in ("title", "anywhere"):
        for n in (0, 3, 5):
            q = field(db, where, SHARING[db]) if n == 0 else share_near_data(db, where, n)
            lim = limit(db) if n == 0 else limit_near(db)
            recs, hits = sl.count(db, f"({q}) AND {lim}"), sl.retrieved(db, q, dev)
            if n == 0:
                phrase[where] = hits
            rows.append(dict(database=db, field=where, sharing="phrase" if n == 0 else f"within {n} words",
                             query=q, limit=lim, records_2015_2025=recs, development_found=len(hits),
                             beyond_phrase="; ".join(sorted(LABEL[r] for r in hits - phrase[where])),
                             studies="; ".join(sorted(LABEL[r] for r in hits))))
sl.write("term_table_sharing_proximity.csv", rows)

##### Exploratory run 15. Candidate queries across the three databases

**Question.** Across the three databases together, which query finds the most development sources for the fewest records?

**Measured.** Records with duplicates removed, and development sources at least one database finds, out of the 35 at least one indexes, for phrase plus factor words and three candidates:

1. share within 3 words of *data*, title;
2. share within 5 words of *data*, title;
3. candidate 2 OR the factor part of phrase plus factor words.

**Result.**

| | Records | Without duplicates | Development sources found |
|:--|--:|--:|--:|
| Phrase plus factor words | 23,488 | 14,255 | 28 |
| 1. Share within 3 words, title | 18,258 | 9,591 | 28 |
| 2. Share within 5 words, title | 19,892 | 10,493 | 29 |
| 3. Candidate 2 OR factor part | 30,985 | 18,135 | 31 |

Candidate 2 adds Zimmerman (2008) for 902 records; candidate 3 adds Fisher and Fortmann (2010) and Piwowar (2011) for 7,642 more. Every query misses Cohn (2012), Costello (2009), McCullough (2009) and Parr (2007). Their titles speak of open sourcing, online publication, open access journals and DataONE.

**Decision.** Phrase plus factor words drops out, since candidate 1 finds as many sources in about 4,700 fewer records. We chose candidate 2. The two sources candidate 3 adds do not justify 7,642 more records, so the factor terms go to screening.

Files: [candidate_queries.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/candidate_queries.csv).

In [19]:
#| eval: false
in_any = {s["ref_id"] for db in DATABASES for s in dev_in(db) if sl.count(db, sl.id_clause(db, [s])) > 0}


def factor_part(db):
    """The factor part of version 3: anywhere in PubMed, within 5 words in Scopus and Web of Science."""
    s, f = " OR ".join(SHARING[db]), " OR ".join(FACTORS)
    if db == "PubMed":
        return f"({field(db, 'anywhere', SHARING[db])} AND {field(db, 'anywhere', FACTORS)})"
    return f"TITLE-ABS-KEY(({s}) W/5 ({f}))" if db == "Scopus" else f"TS=(({s}) NEAR/5 ({f}))"


CANDIDATES = {"1 title, within 3 words": lambda db: share_near_data(db, "title", 3),
              "2 title, within 5 words": lambda db: share_near_data(db, "title", 5),
              "3 version 2 OR factor part": lambda db: f"({share_near_data(db, 'title', 5)} OR {factor_part(db)})"}
summary, all_recs = [], []
for name, make in CANDIDATES.items():
    found, recs = set(), []
    for db in DATABASES:
        q = make(db)
        hits = sl.retrieved(db, q, dev_in(db))
        got = fetch(db, q, limit_near(db))
        recs += [dict(r, database=db, candidate=name) for r in got]
        found |= hits
        summary.append(dict(candidate=name, database=db, query=q, limit=limit_near(db), records_2015_2025=len(got),
                            development_found=len(hits), studies="; ".join(sorted(LABEL[r] for r in hits))))
    summary.append(dict(candidate=name, database="all three", records_2015_2025=len(recs),
                        unique_records=unique_count(recs), development_found=len(found), of_indexed=len(in_any),
                        missed_indexed="; ".join(sorted(LABEL[r] for r in in_any - found))))
    all_recs += recs
sl.write("candidate_queries.csv", summary)
sl.write("candidate_records.csv", all_recs)

##### Exploratory run 16. Document types

**Question.** Could a document type limit narrow candidate 2 without dropping sources the review keeps?

**Measured.** Candidate 2's records by each database's document type, and the database's type for each development source it finds, beside the source type the coding gave it.

**Result.** Editorials, letters, notes and news make up about 18% of the records in PubMed, 8% in Scopus and 11% in Web of Science. The databases' types do not follow the coded source types. Most opinion pieces are indexed as Article or Review, and a Note or Editorial can be a policy document (Harding et al. 2013) or an empirical study (Huang et al. 2013).

**Decision.** No document type limit. A limit would keep most opinion pieces and drop sources the review keeps, and the protocol makes all source types eligible.

Files: [document_types.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/document_types.csv).

In [20]:
#| eval: false
import os
import collections

Q2 = {db: share_near_data(db, "title", 5) for db in DATABASES}
rows = []


def record(db, counts, total, known):
    for t, n in counts.items():
        rows.append(dict(database=db, what="records", document_type=t, n=n, of=total))
    for r, t in known.items():
        rows.append(dict(database=db, what="development study", document_type=t, label=LABEL[r],
                         source_type=next(s["source_type"] for s in DEV if s["ref_id"] == r)))


# PubMed: publication types overlap, so each count is the records carrying that type
db, q = "PubMed", f"({Q2['PubMed']}) AND {limit('PubMed')}"
pm_types = ["Journal Article", "Review", "Editorial", "Comment", "Letter", "News", "Published Erratum",
            "Practice Guideline", "Guideline", "Systematic Review", "Meta-Analysis"]
counts = {t: sl.count(db, f'{q} AND "{t}"[pt]') for t in pm_types}
counts["none of Editorial, Comment, Letter, News"] = sl.count(db, f"{q} NOT (Editorial[pt] OR Comment[pt] OR Letter[pt] OR News[pt])")
known = {}
for r in sl.retrieved(db, Q2[db], dev_in(db)):
    _, ids = sl.pubmed_search(sl.id_clause(db, [s for s in DEV if s["ref_id"] == r]), retmax=1)
    j = sl._get(sl.EU + "esummary.fcgi", dict(db="pubmed", id=ids[0], retmode="json",
                api_key=os.environ["NCBI_API_KEY"]), pause=0.11).json()["result"][ids[0]]
    known[r] = ", ".join(j.get("pubtype", []))
record(db, counts, sl.count(db, q), known)

# Scopus: one document type per record
db, q = "Scopus", f"({Q2['Scopus']}) AND {limit('Scopus')}"
sc_types = {"ar": "Article", "re": "Review", "ed": "Editorial", "le": "Letter", "no": "Note", "sh": "Short survey",
            "cp": "Conference paper", "ch": "Book chapter", "bk": "Book", "er": "Erratum", "cr": "Conference review",
            "dp": "Data paper"}
counts = {name: sl.count(db, f"{q} AND DOCTYPE({code})") for code, name in sc_types.items()}
known = {}
for r in sl.retrieved(db, Q2[db], dev_in(db)):
    doi = next(s["doi"] for s in DEV if s["ref_id"] == r)
    j = sl._get(sl.SC, dict(query=f"DOI({doi})", field="doi,subtypeDescription"),
                {"X-ELS-APIKey": os.environ["SCOPUS_API_KEY"], "Accept": "application/json"}, pause=0.12).json()
    known[r] = j["search-results"]["entry"][0].get("subtypeDescription", "?")
record(db, counts, sl.count(db, q), known)

# Web of Science: types read from every record; a record can carry more than one
db, q = "Web of Science", f"({Q2['Web of Science']}) AND {limit_near('Web of Science')}"
counts, total, page = collections.Counter(), 0, 1
while True:
    j = sl.wos_search(q, limit=50, page=page)
    for h in j.get("hits", []):
        counts.update(h.get("types", ["?"]))
        total += 1
    if page * 50 >= j["metadata"]["total"] or not j.get("hits"):
        break
    page += 1
known = {}
for r in sl.retrieved(db, Q2[db], dev_in(db)):
    doi = next(s["doi"] for s in DEV if s["ref_id"] == r)
    h = sl.wos_search(f"DO=({doi})", limit=1)["hits"]
    known[r] = ", ".join(h[0].get("types", ["?"])) if h else "?"
record(db, dict(counts), total, known)
sl.write("document_types.csv", rows)

#### Scope

Opening tier 2 to all source types adds about a third more full-text screening, and we opened it for the comparison the third question asks for (exploratory run 18). Exploratory runs 17 and 18 measure what opening tier 2 to all source types would cost in screening. We ran them on 18 November 2026. They use the records of the title query. The title query searches titles for share within 5 words of data and leaves out titles that name computing systems.


##### Exploratory run 17. Tier 2: the records that could be included

**Question.** The protocol limited tier 2, research in other fields and cross-disciplinary research, to reviews and syntheses. How many records does that limit keep out?

**Measured.** Three counts that do not judge relevance: reference set sources that meet the criteria with and without the limit, the share of the title query's records in PubMed, and Scopus's split of its records by subject area. Scopus applies AND NOT after every AND, so its limits come before the title query's NOT clause.

**Result.** In the reference set, 30 sources meet every criterion apart from year with the limit and 76 without. Of the title query's 7,933 records, 30% are in PubMed. Scopus splits its 7,067 records into 2,196 in health (31%), 84 reviews in other fields (1%) and 4,787 other records in other fields (68%).

**Decision.** The limit keeps out most records in other fields. Exploratory run 18 estimates how many of them would be included.

In [21]:
#| eval: false
BASE = "TITLE(shar* W/5 data) AND PUBYEAR > 2014 AND PUBYEAR < 2026 AND LANGUAGE(english OR french)"
NOT = 'TITLE(blockchain* OR "smart contract*" OR IoT OR "internet of things" OR vehicular OR "edge computing" OR cloud)'
HEALTH = "SUBJAREA(MEDI OR NURS OR HEAL OR DENT OR PHAR OR IMMU OR NEUR OR BIOC)"  # Scopus's health subject areas


def n(extra=""):
    return sl.count("Scopus", f"(({BASE}) {extra}) AND NOT {NOT}")


total, health = n(), n(f"AND {HEALTH}")
other_reviews = n("AND DOCTYPE(re)") - n(f"AND DOCTYPE(re) AND {HEALTH}")
print(total, health, other_reviews, total - health - other_reviews)

##### Exploratory run 18. Tier 2: the records that would be included

**Question.** How many more records would full-text screening take if tier 2 admitted all source types?

**Measured.** The local model judged 300 of the title query's 7,933 records, drawn at random with seed 20261118, with `scope_prompt.md`. The prompt states the screening guide's criteria without the tier criterion and asks for the field and whether the record is a review. Abstracts came from PubMed, Scopus or OpenAlex; 39 records had none. We discarded a first pass with `coding_prompt.md`. That prompt does not state this review's criteria, and the model counted computing systems and company data as research data sharing.

**Result.**

| Records the model includes | Of 300 | Share (95% interval) | Projected to 7,933 |
|:--|--:|:--|--:|
| Tier 2 limited to reviews and syntheses | 48 | 16% (12% to 21%) | 1,270 |
| Tier 2 open to all source types | 64 | 21% (17% to 26%) | 1,690 |
| Only with tier 2 open | 16 | 5% (3% to 9%) | 420 |

31 records are undecidable, 28 of them judged on title alone; counted as included, opening tier 2 adds 35 records, 930 projected. Most of the 16 concern factors in research data sharing outside health, such as "Data sharing in PLOS ONE: An analysis of Data Availability Statements".

**Decision.** We opened tier 2 to all source types and accepted about a third more full-text screening, for the comparison the third question asks for.

Files: [scope_sample_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/scope_sample_records.csv), [scope_screening_qwen3.5-122b.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/scope_screening_qwen3.5-122b.csv).

In [22]:
#| eval: false
# On the analysis server, with the local model running (scripts/launch_rds_llm.sh) and the sample drawn:
#   python3 analysis/00_search-development/local-model/coding.py scope_codes_qwen3.5-122b.csv scope_sample_records.csv
#   python3 analysis/00_search-development/local-model/scope.py scope_screening_qwen3.5-122b.csv
# The sample: one record per work from the query's records, drawn at random with seed 20261118.
import random

key, works = {}, []
order = {"PubMed": 0, "Scopus": 1, "Web of Science": 2}
recs = [r for r in sl.read("query_exclusion_records.csv") if r["version"] == "+ cloud"]
for r in sorted(recs, key=lambda r: order[r["database"]]):
    ks = [k for k in (("d", r["doi"]) if r["doi"] else None,
                      ("t", re.sub(r"[^a-z0-9]", "", r["title"].lower()) + r["year"]) if r["title"] else None) if k]
    g = next((key[k] for k in ks if k in key), None)
    if g is None:
        g = len(works)
        works.append(r)
    for k in ks:
        key[k] = g
sample = random.Random(20261118).sample(works, 300)  # abstracts are then fetched from PubMed, Scopus and OpenAlex

#### The title query

The title query found 18 of 25 indexed test sources. Seven of those it missed name research data management, open data or open science in their titles (exploratory run 19). The title query searches titles for a form of *share* within 5 words of *data*, and leaves out titles that name *blockchain\**, *"smart contract\*"*, *IoT*, *"internet of things"*, *vehicular*, *"edge computing"* or *cloud*, for 2015 to 2025 in English or French. It returns 7,933 records once duplicates count as one. We ran exploratory run 19 on 18 November 2026. The entries for 18 November 2026 in `log.md` record how we chose the words left out. The revised query replaces it (exploratory runs 20 to 27).


##### Exploratory run 19. The title query against the test set

**Question.** How much of the held-out test set does the title query find?

**Measured.** The title query against the 31 test sources, checked by DOI or PMID in each database, with a Wilson 95% interval. No earlier run used the test set to choose a query.

**Result.**

| | Found | 95% interval |
|:--|--:|:--|
| PubMed | 8 of 11 | 0.43 to 0.90 |
| Scopus | 16 of 23 | 0.49 to 0.84 |
| Web of Science | 12 of 18 | 0.44 to 0.84 |
| At least one database | 18 of 25 | 0.52 to 0.86 |

The title query misses 13 test sources. No query can find six: Bishoff and Johnston (2015), He (2016) and Zvyagintseva (2015) are in none of the databases, and Brown et al. (2015), Guédon (2015) and Koopman (2015) have no DOI or PMID. The other seven name research data management, open data or open science in their titles. These figures count every eligible test source, since tier 2 now admits all source types (exploratory runs 17 and 18). With tier 2 limited, 10 test sources were in scope and the title query found 9.

**Decision.** Exploratory runs 20 to 25 revise the query, searching titles and abstracts.

Files: [test_coverage.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/test_coverage.csv), [test_missed.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/test_missed.csv).

In [23]:
#| eval: false
QUERY_NOT = BLOCKCHAIN + DEVICES + ["cloud"]
test = [s for s in CODED if s["set"] == "test" and s["eligible_apart_from_year"] == "yes"]
found, indexed, rows = {}, {}, []
for db in DATABASES:
    t = [s for s in test if sl.testable(db, s)]
    indexed[db] = {s["ref_id"] for s in t if sl.count(db, sl.id_clause(db, [s])) > 0}
    found[db] = sl.retrieved(db, share_not(db, QUERY_NOT), t)
    p, lo, hi = sl.wilson(len(found[db]), len(indexed[db]))
    rows.append(dict(database=db, query=share_not(db, QUERY_NOT), found=len(found[db]), indexed=len(indexed[db]),
                     coverage=p, low=lo, high=hi))
any_found, in_any = set().union(*found.values()), set().union(*indexed.values())
p, lo, hi = sl.wilson(len(any_found), len(in_any))
rows.append(dict(database="all three", found=len(any_found), indexed=len(in_any), coverage=p, low=lo, high=hi))
missed = []
for s in sorted(test, key=lambda s: s["label"]):
    if s["ref_id"] in any_found:
        continue
    where = {db: ("no DOI or PMID" if not (s["doi"] or s["pmid"]) else
                  "not indexed" if s["ref_id"] not in indexed[db] else "indexed, not found") for db in DATABASES}
    missed.append(dict(ref_id=s["ref_id"], label=s["label"], title=s["title"], year=s["year"], doi=s["doi"],
                       pmid=s["pmid"], field=s["field"], source_type=s["source_type"], in_scope=s["in_scope"],
                       pubmed=where["PubMed"], scopus=where["Scopus"], web_of_science=where["Web of Science"]))
sl.write("test_coverage.csv", rows)
sl.write("test_missed.csv", missed)

#### Revising the query

A search of titles and abstracts with a research term replaced the title query, and the NOT clause dropped out (exploratory runs 20 to 25). The title query misses seven indexed test sources (exploratory run 19). Exploratory runs 20 to 25 revise it to search titles and abstracts, and test a research term to bring the records down. We ran them, and exploratory runs 26 and 27, on 23 November 2026. We name every search by its terms. These runs count the sources each search loses among the development and test sources together, so the test set did not stay out of choosing the revised query, and exploratory run 26 does not test it on sources independent of that choice. The title query left out titles that name computing systems. Exploratory runs 23 to 25 check what these words remove from titles and abstracts.


##### Exploratory run 20. Health terms

**Question.** Do terms common in health research find known sources that share near data misses?

**Measured.** Three terms in titles and abstracts, each alone: *"individual participant data"*, *"individual patient data"* and *"clinical trial data"*. For each, the records it returns for 2015 to 2025, the records it adds to share near data in titles and abstracts with NOT (*blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* OR *vehicular* OR *"edge computing"* OR *cloud*) applied to titles, and the development and test sources it finds that share near data misses.

**Result.** None of the three finds a known source share near data misses. Each adds 2,180 to 5,056 records per database.

**Decision.** The health terms stay out of the query.

Files: [term_table_research.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_research.csv), [term_table_research_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_research_console.txt).

In [24]:
#| eval: false
# Research and health terms, with NOT (blockchain* ... cloud) applied to titles. Writes term_table_research.csv.
NOT_TERMS = ["blockchain*", '"smart contract*"', "IoT", '"internet of things"', "vehicular", '"edge computing"', "cloud"]
CONTEXT = ["research*", "scientific", "scientist*", "study", "studies", "trial*", "cohort*"]
CONTEXT_NO_STUDY = [w for w in CONTEXT if w not in ("study", "studies")]
HEALTH = ['"individual participant data"', '"individual patient data"', '"clinical trial data"']
TEST = [s for s in CODED if s["set"] == "test" and s["eligible_apart_from_year"] == "yes"]


def research_near(db, where):
    """A form of share within 5 words of "research data"."""
    if db == "PubMed":
        tag = "ti" if where == "title" else "tiab"
        return "(" + " OR ".join(f'"{w} research data"[{tag}:~5]' for w in FORMS) + ")"
    t = {"Scopus": {"title": "TITLE", "anywhere": "TITLE-ABS-KEY"},
         "Web of Science": {"title": "TI=", "anywhere": "TS="}}[db][where]
    return f'{t}(shar* W/5 "research data")' if db == "Scopus" else f'{t}(shar* NEAR/5 "research data")'


def both(a, b):
    return f"({a} AND {b})"


def minus(db, a, b):
    """Records a finds that b does not."""
    return f"({a} AND NOT {b})" if db == "Scopus" else f"({a} NOT {b})"


def with_not(db, q):
    return minus(db, q, field(db, "title", NOT_TERMS))


def queries(db):
    title, anywhere = share_near_data(db, "title", 5), share_near_data(db, "anywhere", 5)
    return [("title", "baseline: share near data", title),
            ("anywhere", "baseline: share near data", anywhere),
            ("anywhere", 'research 1: share near "research data"', research_near(db, "anywhere")),
            ("anywhere", "research 2: share near data AND research*", both(anywhere, field(db, "anywhere", ["research*"]))),
            ("anywhere", "research 3: share near data AND a research word", both(anywhere, field(db, "anywhere", CONTEXT))),
            ("anywhere", "research 4: research 3 without study, studies", both(anywhere, field(db, "anywhere", CONTEXT_NO_STUDY)))]


def row(**kw):
    base = dict(database="", field="", test="", version="", NOT="", query="", records_2015_2025="",
                records_added_beyond_baseline="", development_found="", development_of="", test_found="", test_of="",
                lost_from_baseline="", beyond_baseline="")
    base.update(kw)
    return base


rows = []
for db in DATABASES:
    dev = [s for s in DEV if sl.testable(db, s)]
    test = [s for s in TEST if sl.testable(db, s)]
    baseline = {}
    for where, name, q0 in queries(db):
        for nt in ("with NOT", "without NOT"):
            q = with_not(db, q0) if nt == "with NOT" else q0
            recs = sl.count(db, f"({q}) AND {limit_near(db)}")
            d, t = sl.retrieved(db, q, dev), sl.retrieved(db, q, test)
            if name.startswith("baseline") and nt == "with NOT":
                baseline[where] = (q, d | t)
            lost = baseline["anywhere"][1] - d - t if not name.startswith("baseline") else set()
            rows.append(row(database=db, field=where, test="research", version=name, NOT=nt, query=q,
                            records_2015_2025=recs, development_found=len(d), development_of=len(dev),
                            test_found=len(t), test_of=len(test),
                            lost_from_baseline="; ".join(sorted(LABEL.get(r, r) for r in lost))))
    base_q, base_found = baseline["anywhere"]
    for term in HEALTH:
        q = field(db, "anywhere", [term])
        recs = sl.count(db, f"({q}) AND {limit_near(db)}")
        added = sl.count(db, f"({minus(db, q, base_q)}) AND {limit_near(db)}")
        d, t = sl.retrieved(db, q, dev), sl.retrieved(db, q, test)
        beyond = (d | t) - base_found
        rows.append(row(database=db, field="anywhere", test="health", version=term, query=q, records_2015_2025=recs,
                        records_added_beyond_baseline=added, development_found=len(d), development_of=len(dev),
                        test_found=len(t), test_of=len(test),
                        beyond_baseline="; ".join(sorted(LABEL.get(r, r) for r in beyond))))
    sl.write("term_table_research.csv", rows)

##### Exploratory run 21. Research terms

**Question.** With every part of the query in titles and abstracts, which research term cuts the most records for the fewest sources lost?

**Measured.** Five searches in titles and abstracts. They are share near data; share near *"research data"*; share near data AND *research\**; share near data AND (*research\** OR *scientific* OR *scientist\** OR *study* OR *studies* OR *trial\** OR *cohort\**); share near data AND (*research\** OR *scientific* OR *scientist\** OR *trial\** OR *cohort\**); each with and without NOT (*blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* OR *vehicular* OR *"edge computing"* OR *cloud*), now applied to titles and abstracts. Sources lost count against share near data under the row's NOT setting.

**Result.** With the NOT; each cell gives records, the share of records cut from share near data, and sources lost.

| Search | PubMed | Scopus | Web of Science |
|:--|:--|:--|:--|
| share near data | 17,790 | 62,769 | 49,700 |
| share near *"research data"* | 1,469 · 92% · 15 | 686 · 99% · 41 | 553 · 99% · 37 |
| share near data AND *research\** | 8,151 · 54% · 5 | 24,191 · 61% · 8 | 18,216 · 63% · 8 |
| share near data AND (*research\** OR *scientific* OR *scientist\** OR *study* OR *studies* OR *trial\** OR *cohort\**) | 13,741 · 23% · 1 | 40,641 · 35% · 2 | 32,259 · 35% · 3 |
| share near data AND (*research\** OR *scientific* OR *scientist\** OR *trial\** OR *cohort\**) | 10,003 · 44% · 1 | 27,420 · 56% · 3 | 21,041 · 58% · 3 |

The NOT loses no source in any search. On share near data it removes 6% of records in PubMed, 24% in Scopus and 20% in Web of Science.

**Decision.** Exploratory run 22 tests *scienti\** as a shorter form of *scientific* and *scientist\**, without *trial\** and *cohort\**.

Files: [research_terms.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/research_terms.csv), [research_terms_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/research_terms_console.txt).

In [25]:
#| eval: false
# Research terms, each with and without NOT (blockchain* ... cloud) in titles and abstracts. Writes research_terms.csv.
NOT_TERMS = ["blockchain*", '"smart contract*"', "IoT", '"internet of things"', "vehicular", '"edge computing"', "cloud"]
WORDS = ["research*", "scientific", "scientist*", "study", "studies", "trial*", "cohort*"]
WORDS_NO_STUDY = [w for w in WORDS if w not in ("study", "studies")]
TEST = [s for s in CODED if s["set"] == "test" and s["eligible_apart_from_year"] == "yes"]


def research_near(db):
    """A form of share within 5 words of "research data", in titles and abstracts."""
    if db == "PubMed":
        return "(" + " OR ".join(f'"{w} research data"[tiab:~5]' for w in FORMS) + ")"
    return 'TITLE-ABS-KEY(shar* W/5 "research data")' if db == "Scopus" else 'TS=(shar* NEAR/5 "research data")'


def with_not(db, q):
    """Leaves out records that name a computing word in the title or abstract."""
    t = field(db, "anywhere", NOT_TERMS)
    return f"({q} AND NOT {t})" if db == "Scopus" else f"({q} NOT {t})"


def searches(db):
    base = share_near_data(db, "anywhere", 5)
    return [("share near data", base),
            ('share near "research data"', research_near(db)),
            ("share near data AND research*", f"({base} AND {field(db, 'anywhere', ['research*'])})"),
            ("share near data AND a research word", f"({base} AND {field(db, 'anywhere', WORDS)})"),
            ("share near data AND a research word without study and studies",
             f"({base} AND {field(db, 'anywhere', WORDS_NO_STUDY)})")]


rows = []
for db in DATABASES:
    dev = [s for s in DEV if sl.testable(db, s)]
    test = [s for s in TEST if sl.testable(db, s)]
    reference = {}
    for name, q0 in searches(db):
        for nt in ("without NOT", "with NOT"):
            q = with_not(db, q0) if nt == "with NOT" else q0
            recs = sl.count(db, f"({q}) AND {limit_near(db)}")
            d, t = sl.retrieved(db, q, dev), sl.retrieved(db, q, test)
            if name == "share near data":
                reference[nt] = d | t
            lost = reference[nt] - d - t
            rows.append(dict(database=db, search=name, NOT=nt, query=q, records_2015_2025=recs,
                             development_found=len(d), development_of=len(dev), test_found=len(t), test_of=len(test),
                             lost_against_share_near_data="; ".join(sorted(LABEL.get(r, r) for r in lost))))
    sl.write("research_terms.csv", rows)

##### Exploratory run 22. share near data AND (research\* OR scienti\*)

**Question.** Does *scienti\**, which matches scientific, scientist and scientists but not science, recover the sources *research\** alone loses, and what do *trial\** and *cohort\** add?

**Measured.** share near data AND (*research\** OR *scienti\**), in titles and abstracts, with and without NOT (*blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* OR *vehicular* OR *"edge computing"* OR *cloud*) applied to titles and abstracts, with the sources lost against share near data by name.

**Result.** With the NOT; each cell gives records and sources lost.

| Search | PubMed | Scopus | Web of Science |
|:--|:--|:--|:--|
| share near data AND *research\** | 8,151 · 5 | 24,191 · 8 | 18,216 · 8 |
| share near data AND (*research\** OR *scienti\**) | 8,706 · 3 | 25,745 · 3 | 19,501 · 5 |
| share near data AND (*research\** OR *scientific* OR *scientist\** OR *trial\** OR *cohort\**) | 10,003 · 1 | 27,420 · 3 | 21,041 · 3 |

share near data AND (*research\** OR *scienti\**) loses Mennes et al. (2013) in every database, Houtkoop et al. (2018) and Huang et al. (2013) in Scopus and Web of Science, and Mbuagbaw et al. (2017) and Savage and Vickers (2009) in PubMed and Web of Science. *trial\** and *cohort\** recover only Mbuagbaw et al. (2017) and Savage and Vickers (2009), and Scopus finds both without them, so across the three databases *trial\** and *cohort\** add no source for about 4,500 records.

**Decision.** The research term is (*research\** OR *scienti\**).

Files: [research_terms_scienti.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/research_terms_scienti.csv), [research_terms_scienti_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/research_terms_scienti_console.txt).

In [26]:
#| eval: false
# share near data AND (research* OR scienti*), with and without NOT (blockchain* ... cloud) in titles and abstracts.
# Writes research_terms_scienti.csv.
NOT_TERMS = ["blockchain*", '"smart contract*"', "IoT", '"internet of things"', "vehicular", '"edge computing"', "cloud"]
TEST = [s for s in CODED if s["set"] == "test" and s["eligible_apart_from_year"] == "yes"]


def with_not(db, q):
    """Leaves out records that name a computing word in the title or abstract."""
    t = field(db, "anywhere", NOT_TERMS)
    return f"({q} AND NOT {t})" if db == "Scopus" else f"({q} NOT {t})"


def searches(db):
    base = share_near_data(db, "anywhere", 5)
    return [("share near data", base),
            ("share near data AND (research* OR scienti*)",
             f"({base} AND {field(db, 'anywhere', ['research*', 'scienti*'])})")]


rows = []
for db in DATABASES:
    dev = [s for s in DEV if sl.testable(db, s)]
    test = [s for s in TEST if sl.testable(db, s)]
    reference = {}
    for name, q0 in searches(db):
        for nt in ("without NOT", "with NOT"):
            q = with_not(db, q0) if nt == "with NOT" else q0
            recs = sl.count(db, f"({q}) AND {limit_near(db)}")
            d, t = sl.retrieved(db, q, dev), sl.retrieved(db, q, test)
            if name == "share near data":
                reference[nt] = d | t
            lost = sorted(LABEL.get(r, r) for r in reference[nt] - d - t)
            rows.append(dict(database=db, search=name, NOT=nt, query=q, records_2015_2025=recs,
                             development_found=len(d), development_of=len(dev), test_found=len(t), test_of=len(test),
                             lost_against_share_near_data="; ".join(lost)))
    sl.write("research_terms_scienti.csv", rows)

##### Exploratory run 23. The records the NOT removes

**Question.** What records does NOT (*blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* OR *vehicular* OR *"edge computing"* OR *cloud*) remove from share near data AND (*research\** OR *scienti\**), in titles and abstracts?

**Measured.** The records each word matches, every record the NOT removes, and 20 of them per database drawn at random with seed 20261123.

**Result.** The NOT removes 488 records in PubMed, 5,938 in Scopus and 3,262 in Web of Science. *cloud* and *blockchain\** match the most. Of the 40 sampled titles from Scopus and Web of Science, 39 describe computing systems. Of the 20 from PubMed, most describe systems, and three concern research data sharing itself: "National and Transnational Security Implications of Asymmetric Access to and Use of Biological Data", "Connecting data, tools and people across Europe: ELIXIR's response to the COVID-19 pandemic" and a systematic review of challenges in building a digital health data integration ecosystem.

**Decision.** Exploratory run 24 tests the NOT without *cloud*, the word most likely to name research infrastructure.

Files: [not_removed_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/not_removed_records.csv), [not_removed_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/not_removed_console.txt).

In [27]:
#| eval: false
# The records NOT (blockchain* ... cloud) removes from share near data AND (research* OR scienti*), in titles and
# abstracts: the count for each word, every removed record, and 20 at random per database. Writes not_removed_records.csv.
import random

NOT_TERMS = ["blockchain*", '"smart contract*"', "IoT", '"internet of things"', "vehicular", '"edge computing"', "cloud"]
random.seed(20261123)

rows = []
for db in DATABASES:
    q = f"({share_near_data(db, 'anywhere', 5)} AND {field(db, 'anywhere', ['research*', 'scienti*'])})"
    lim = limit_near(db)
    for term in NOT_TERMS:
        print(db, term, sl.count(db, f"({q} AND {field(db, 'anywhere', [term])}) AND {lim}"))
    removed = fetch(db, f"({q} AND {field(db, 'anywhere', NOT_TERMS)})", lim)
    rows += [dict(database=db, id=r["id"], doi=r["doi"], year=r["year"], title=r["title"]) for r in removed]
    for r in random.sample(removed, min(20, len(removed))):
        print(f"    {r['year']}  {r['title'][:150]}")
    sl.write("not_removed_records.csv", rows)

##### Exploratory run 24. The NOT without cloud

**Question.** How much of the NOT's work does *cloud* do, and what do the records only *cloud* removes concern?

**Measured.** On share near data AND (*research\** OR *scienti\**), in titles and abstracts, the records removed by NOT (*blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* OR *vehicular* OR *"edge computing"* OR *cloud*), by the same NOT without *cloud*, and by NOT (*blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"*); and the records that name *cloud* and none of the other six words, with 20 per database drawn at random.

**Result.**

| NOT | PubMed | Scopus | Web of Science |
|:--|--:|--:|--:|
| *blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* OR *vehicular* OR *"edge computing"* OR *cloud* | 488 | 5,938 | 3,262 |
| *blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* OR *vehicular* OR *"edge computing"* | 285 | 4,274 | 2,258 |
| *blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* | 276 | 4,033 | 2,118 |

*cloud* alone removes 203 records in PubMed, 1,664 in Scopus and 1,004 in Web of Science. The PubMed sample includes "Population Neuroscience: Strategies to Promote Data Sharing While Protecting Privacy" and "Share all SARS-CoV-2 data immediately", and the Scopus sample "Towards a European health research and innovation cloud (HRIC)". *vehicular* and *"edge computing"* together remove 9 records in PubMed, 241 in Scopus and 140 in Web of Science beyond the other four words.

**Decision.** *cloud*, *vehicular* and *"edge computing"* leave the NOT. Exploratory run 25 measures each of the four words left.

Files: [cloud_only_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/cloud_only_records.csv), [not_narrowed_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/not_narrowed_console.txt).

In [28]:
#| eval: false
# Three NOT clauses on share near data AND (research* OR scienti*), in titles and abstracts, and the records only cloud
# removes, with 20 at random per database. Writes cloud_only_records.csv.
import random

SEVEN = ["blockchain*", '"smart contract*"', "IoT", '"internet of things"', "vehicular", '"edge computing"', "cloud"]
SIX = [t for t in SEVEN if t != "cloud"]
FOUR = ["blockchain*", '"smart contract*"', "IoT", '"internet of things"']
random.seed(20261123)


def minus(db, a, b):
    return f"({a} AND NOT {b})" if db == "Scopus" else f"({a} NOT {b})"


rows = []
for db in DATABASES:
    q = f"({share_near_data(db, 'anywhere', 5)} AND {field(db, 'anywhere', ['research*', 'scienti*'])})"
    lim = limit_near(db)
    total = sl.count(db, f"({q}) AND {lim}")
    for terms in (SEVEN, SIX, FOUR):
        print(db, len(terms), total - sl.count(db, f"({minus(db, q, field(db, 'anywhere', terms))}) AND {lim}"))
    cloud_only = fetch(db, minus(db, f"({q} AND {field(db, 'anywhere', ['cloud'])})", field(db, "anywhere", SIX)), lim)
    rows += [dict(database=db, id=r["id"], doi=r["doi"], year=r["year"], title=r["title"]) for r in cloud_only]
    for r in random.sample(cloud_only, min(20, len(cloud_only))):
        print(f"    {r['year']}  {r['title'][:150]}")
    sl.write("cloud_only_records.csv", rows)

##### Exploratory run 25. What each of blockchain\*, "smart contract\*", IoT and "internet of things" removes

**Question.** Does each of the four words remove records the others do not, and what do they concern?

**Measured.** On share near data AND (*research\** OR *scienti\**), in titles and abstracts, the records each word matches that none of the other three does, with 10 per word and database drawn at random.

**Result.**

| Word | PubMed | Scopus | Web of Science |
|:--|--:|--:|--:|
| *blockchain\** | 129 | 1,401 | 706 |
| *"smart contract\*"* | 0 | 19 | 8 |
| *IoT* | 10 | 193 | 125 |
| *"internet of things"* | 17 | 301 | 166 |

*"smart contract\*"* removes almost nothing the other words do not. *blockchain\** removes "Role of Incentives in the Use of Blockchain-Based Platforms for Sharing Sensitive Health Data: Experimental Study" in PubMed and Scopus. That record studies why people use a sharing system, and the eligibility criteria admit it. *"internet of things"* in abstracts removes commentaries on health data such as "Susan G. Komen Big Data for Breast Cancer Initiative: How Patient Advocacy Organizations Can Facilitate Using Big Data to Improve Patient Outcomes" and "Digital technologies: tensions in privacy and data". The NOT loses no known source, but in titles and abstracts each of its words also removes eligible records.

**Decision.** The query drops the NOT. Without it, share near data AND (*research\** OR *scienti\**) returns 9,194 records in PubMed, 31,683 in Scopus and 22,763 in Web of Science, 6,427 more across the three databases than with NOT (*blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"*), and loses no known source.

Files: [not_unique_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/not_unique_records.csv), [not_unique_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/not_unique_console.txt).

In [29]:
#| eval: false
# The records each of blockchain*, "smart contract*", IoT and "internet of things" removes that none of the other three
# does, from share near data AND (research* OR scienti*), in titles and abstracts, with 10 at random. Writes
# not_unique_records.csv.
import random

FOUR = ["blockchain*", '"smart contract*"', "IoT", '"internet of things"']
random.seed(20261123)


def minus(db, a, b):
    return f"({a} AND NOT {b})" if db == "Scopus" else f"({a} NOT {b})"


rows = []
for db in DATABASES:
    q = f"({share_near_data(db, 'anywhere', 5)} AND {field(db, 'anywhere', ['research*', 'scienti*'])})"
    lim = limit_near(db)
    for term in FOUR:
        others = [t for t in FOUR if t != term]
        only = minus(db, f"({q} AND {field(db, 'anywhere', [term])})", field(db, "anywhere", others))
        n = sl.count(db, f"({only}) AND {lim}")
        print(db, term, n)
        if n:
            recs = fetch(db, only, lim)
            rows += [dict(database=db, word=term, id=r["id"], doi=r["doi"], year=r["year"], title=r["title"]) for r in recs]
            for r in random.sample(recs, min(10, len(recs))):
                print(f"    {r['year']}  {r['title'][:150]}")
    sl.write("not_unique_records.csv", rows)

#### The revised query

A form of *share* within 5 words of *data* AND (*research\** OR *scienti\**), in titles and abstracts, with no NOT clause, 2015 to 2025, English or French. It returns 9,194 records in PubMed, 31,683 in Scopus and 22,763 in Web of Science, about 34,305 once duplicates count as one (exploratory run 27), and finds 20 of the 25 indexed test sources (exploratory run 26).

| Database | Query |
|:--|:--|
| PubMed | ("sharing data"[tiab:~5] OR "shared data"[tiab:~5] OR "share data"[tiab:~5] OR "shares data"[tiab:~5]) AND (research*[tiab] OR scienti*[tiab]) |
| Scopus | TITLE-ABS-KEY(shar* W/5 data) AND (TITLE-ABS-KEY(research*) OR TITLE-ABS-KEY(scienti*)) |
| Web of Science | TS=(shar* NEAR/5 data) AND (TS=(research*) OR TS=(scienti*)) |


##### Exploratory run 26. The revised query against the test set

**Question.** How much of the test set does the revised query find?

**Measured.** share within 5 words of data AND (*research\** OR *scienti\**), in titles and abstracts, against the test sources, checked by DOI or PMID in each database, with a Wilson 95% interval, as in exploratory run 19.

**Result.**

| | Found | 95% interval |
|:--|--:|:--|
| PubMed | 8 of 11 | 0.43 to 0.90 |
| Scopus | 19 of 23 | 0.63 to 0.93 |
| Web of Science | 12 of 18 | 0.44 to 0.84 |
| At least one database | 20 of 25 | 0.61 to 0.91 |

The revised query finds 20 of the 25 indexed test sources, against 18 for the title query (exploratory run 19). It misses 11 test sources. No query can find six: Bishoff and Johnston (2015), He (2016) and Zvyagintseva (2015) are in none of the databases, and Brown et al. (2015), Guédon (2015) and Koopman (2015) have no DOI or PMID. The query misses five that the databases index. Chiware and Mathe (2016), Specht et al. (2015) and Wittenberg and Elings (2017) concern research data management, Watson (2015) concerns open science, and Houtkoop et al. (2018) is one of the sources the research term loses (exploratory run 22).

Files: [test_coverage_revised.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/test_coverage_revised.csv), [test_missed_revised.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/test_missed_revised.csv), [test_coverage_revised_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/test_coverage_revised_console.txt).

In [30]:
#| eval: false
# The revised query against the test set, as in exploratory run 19. Writes test_coverage_revised.csv and test_missed_revised.csv.
def revised(db):
    """share within 5 words of data AND (research* OR scienti*), in titles and abstracts."""
    return f"({share_near_data(db, 'anywhere', 5)} AND {field(db, 'anywhere', ['research*', 'scienti*'])})"


test = [s for s in CODED if s["set"] == "test" and s["eligible_apart_from_year"] == "yes"]
found, indexed, rows = {}, {}, []
for db in DATABASES:
    t = [s for s in test if sl.testable(db, s)]
    indexed[db] = {s["ref_id"] for s in t if sl.count(db, sl.id_clause(db, [s])) > 0}
    found[db] = sl.retrieved(db, revised(db), t)
    p, lo, hi = sl.wilson(len(found[db]), len(indexed[db]))
    rows.append(dict(database=db, query=revised(db), found=len(found[db]), indexed=len(indexed[db]),
                     coverage=p, low=lo, high=hi))
any_found, in_any = set().union(*found.values()), set().union(*indexed.values())
p, lo, hi = sl.wilson(len(any_found), len(in_any))
rows.append(dict(database="all three", found=len(any_found), indexed=len(in_any), coverage=p, low=lo, high=hi))
missed = []
for s in sorted(test, key=lambda s: s["label"]):
    if s["ref_id"] in any_found:
        continue
    where = {db: ("no DOI or PMID" if not (s["doi"] or s["pmid"]) else
                  "not indexed" if s["ref_id"] not in indexed[db] else "indexed, not found") for db in DATABASES}
    missed.append(dict(ref_id=s["ref_id"], label=s["label"], title=s["title"], year=s["year"], doi=s["doi"],
                       pmid=s["pmid"], field=s["field"], source_type=s["source_type"], in_scope=s["in_scope"],
                       pubmed=where["PubMed"], scopus=where["Scopus"], web_of_science=where["Web of Science"]))
sl.write("test_coverage_revised.csv", rows)
sl.write("test_missed_revised.csv", missed)

##### Exploratory run 27. Records of the revised query

**Question.** How many records does the revised query return across the three databases once duplicates count as one?

**Measured.** Every record share within 5 words of data AND (*research\** OR *scienti\**), in titles and abstracts, returns for 2015 to 2025, with each record's DOI, title and year. Records that share a DOI, or a title and year once spacing and punctuation are removed, count as one, as in exploratory run 15. bib-dedupe also compares authors, journals, volumes and pages and finds as many duplicates or more, so this count is an upper bound.

**Result.**

| | Records | With a DOI |
|:--|--:|--:|
| PubMed | 9,194 | 9,018 |
| Scopus | 31,683 | 29,797 |
| Web of Science | 22,763 | 21,036 |
| All three | 63,640 | |
| Unique | 34,305 | |

34,305 of the 63,640 records are unique (54%), against 10,493 unique records for candidate 2 in exploratory run 15.

Files: [revised_query_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/revised_query_records.csv), [revised_query_records_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/revised_query_records_console.txt).

In [31]:
#| eval: false
# Every record of the revised query, and the records left once those sharing a DOI, or a title and year, count as one.
# Writes revised_query_records.csv.
def revised(db):
    """share within 5 words of data AND (research* OR scienti*), in titles and abstracts."""
    return f"({share_near_data(db, 'anywhere', 5)} AND {field(db, 'anywhere', ['research*', 'scienti*'])})"


recs = []
for db in DATABASES:
    got = fetch(db, revised(db), limit_near(db))
    recs += [dict(r, database=db) for r in got]
    sl.write("revised_query_records.csv", recs)
print(len(recs), unique_count(recs))

#### Open near data

Adding open\* near data found 21 of 25 test sources but brought the unique records to 60,852 (exploratory run 31). The main part replaces it with the phrases "open data" and "research data management" (runs 4 and 5). The query misses test sources that describe the practice as open data. Exploratory runs 28 to 31 test a form of *open* within a set distance of *data* beside share near data, and the distance for both. Exploratory run 28 first repeats the research terms of exploratory run 21 with *scienti\** in every search, so that every search in the comparison uses the same research term. Every search runs in titles and abstracts, with no NOT clause. We ran exploratory runs 28 to 31 on 25 November 2026. They count development and test sources together, so the test set did not stay out of choosing the query.

Exploratory run 31 tests (share NEAR/5 data OR *open\** NEAR/5 data) AND (*research\** OR *scienti\**), in titles and abstracts, 2015 to 2025, English or French.

| Database | Query |
|:--|:--|
| PubMed | ("sharing data"[tiab:~5] OR "shared data"[tiab:~5] OR "share data"[tiab:~5] OR "shares data"[tiab:~5] OR "open data"[tiab:~5] OR "opened data"[tiab:~5] OR "opening data"[tiab:~5] OR "openly data"[tiab:~5] OR "openness data"[tiab:~5]) AND (research*[tiab] OR scienti*[tiab]) |
| Scopus | (TITLE-ABS-KEY(shar* W/5 data) OR TITLE-ABS-KEY(open* W/5 data)) AND (TITLE-ABS-KEY(research*) OR TITLE-ABS-KEY(scienti*)) |
| Web of Science | (TS=(shar* NEAR/5 data) OR TS=(open* NEAR/5 data)) AND (TS=(research*) OR TS=(scienti*)) |

: The query as written for each database. {tbl-colwidths="[16,84]"}

##### Exploratory run 28. Research terms with scienti\*

**Question.** With *scienti\** in place of *scientific* and *scientist\**, what does each research term cost in records and in known sources?

**Measured.** Six searches in PubMed, Scopus and Web of Science: share near data; share near *"research data"*; share near data AND *research\**; share near data AND (*research\** OR *scienti\**); share near data AND (*research\** OR *scienti\** OR *trial\** OR *cohort\**); share near data AND (*research\** OR *scienti\** OR *study* OR *studies* OR *trial\** OR *cohort\**). For each, the records for 2015 to 2025 and the development and test sources it loses against share near data.

**Result.** Each cell gives records and sources lost.

| Search | PubMed | Scopus | Web of Science |
|:--|:--|:--|:--|
| share near data | 18,840 · 0 | 82,926 · 0 | 62,228 · 0 |
| share near *"research data"* | 1,536 · 15 | 743 · 41 | 589 · 37 |
| share near data AND *research\** | 8,615 · 5 | 29,910 · 8 | 21,331 · 8 |
| share near data AND (*research\** OR *scienti\**) | 9,194 · 3 | 31,681 · 3 | 22,764 · 5 |
| share near data AND (*research\** OR *scienti\** OR *trial\** OR *cohort\**) | 10,517 · 1 | 33,522 · 3 | 24,396 · 3 |
| share near data AND (*research\** OR *scienti\** OR *study* OR *studies* OR *trial\** OR *cohort\**) | 14,399 · 1 | 49,324 · 2 | 37,273 · 3 |

*trial\** and *cohort\** recover Mbuagbaw et al. (2017) and Savage and Vickers (2009) in PubMed and Web of Science. Scopus finds both without them, so across the three databases *trial\** and *cohort\** add no source for 4,796 records. *study* and *studies* add Mennes et al. (2013) in Scopus alone, for 32,561 records more.

**Decision.** The research term stays (*research\** OR *scienti\**).

Files: [research_terms_scienti_all.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/research_terms_scienti_all.csv), [research_terms_scienti_all_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/research_terms_scienti_all_console.txt).

In [ ]:
#| eval: false
# share near data with each research term, in titles and abstracts, with scienti* in every search.
# Writes research_terms_scienti_all.csv.
TEST = [s for s in CODED if s["set"] == "test" and s["eligible_apart_from_year"] == "yes"]
RS = ["research*", "scienti*"]


def share(db):
    return share_near_data(db, "anywhere", 5)


def share_and(db, w):
    return f"({share(db)} AND {field(db, 'anywhere', w)})"


def rnear(db):
    """A form of share within 5 words of "research data", in titles and abstracts."""
    if db == "PubMed":
        return "(" + " OR ".join(f'"{w} research data"[tiab:~5]' for w in FORMS) + ")"
    return 'TITLE-ABS-KEY(shar* W/5 "research data")' if db == "Scopus" else 'TS=(shar* NEAR/5 "research data")'


A = [("share near data", share), ('share near "research data"', rnear),
     ("share near data AND research*", lambda db: share_and(db, ["research*"])),
     ("share near data AND (research* OR scienti*)", lambda db: share_and(db, RS)),
     ("share near data AND (research* OR scienti* OR trial* OR cohort*)",
      lambda db: share_and(db, RS + ["trial*", "cohort*"])),
     ("share near data AND (research* OR scienti* OR study OR studies OR trial* OR cohort*)",
      lambda db: share_and(db, RS + ["study", "studies", "trial*", "cohort*"]))]
rows = []
for db in DATABASES:
    dev = [s for s in DEV if sl.testable(db, s)]
    test = [s for s in TEST if sl.testable(db, s)]
    ref = None
    for name, make in A:
        q = make(db)
        n = sl.count(db, f"({q}) AND {limit_near(db)}")
        d, t = sl.retrieved(db, q, dev), sl.retrieved(db, q, test)
        ref = d | t if ref is None else ref
        lost = "; ".join(sorted(LABEL.get(r, r) for r in ref - d - t))
        rows.append(dict(database=db, search=name, query=q, limit=limit_near(db), records_2015_2025=n,
                         development_found=len(d), development_of=len(dev), test_found=len(t), test_of=len(test),
                         lost_against_share_near_data=lost))
    sl.write("research_terms_scienti_all.csv", rows)

##### Exploratory run 29. open\* near data

**Question.** Does a form of *open* within five words of *data* find known sources that share near data AND (*research\** OR *scienti\**) misses, and at what cost?

**Measured.** Three searches in PubMed, Scopus and Web of Science: (share NEAR/5 data) AND (*research\** OR *scienti\**); (share NEAR/5 data OR *open\** NEAR/5 data) AND (*research\** OR *scienti\**); (share NEAR/5 data) AND (*research\** OR *scienti\** OR *open\** NEAR/5 data). PubMed's proximity search takes no truncation, so there the block writes out *open*, *opened*, *opening*, *openly* and *openness*. For each, the records for 2015 to 2025, the records it adds to the first search, and the development and test sources it finds that the first search misses.

**Result.**

| Search | PubMed | Scopus | Web of Science |
|:--|--:|--:|--:|
| (share NEAR/5 data) AND (*research\** OR *scienti\**) | 9,194 | 31,681 | 22,764 |
| (share NEAR/5 data OR *open\** NEAR/5 data) AND (*research\** OR *scienti\**) | 14,371 | 55,738 | 40,738 |
| (share NEAR/5 data) AND (*research\** OR *scienti\** OR *open\** NEAR/5 data) | 9,519 | 33,347 | 23,987 |

: Records for 2015 to 2025. {tbl-colwidths="[52,16,16,16]"}

*open\** NEAR/5 data beside share near data adds 47,208 records across the three databases and finds McCullough (2009) and Specht et al. (2015). Beside the research terms, it adds 3,214 records and finds Houtkoop et al. (2018) and Mennes et al. (2013). Neither search loses a source the first search finds.

**Decision.** *open\** NEAR/5 data joins share NEAR/5 data. Beside the research terms, it would admit records on data sharing outside research.

Files: [open_terms.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/open_terms.csv), [open_terms_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/open_terms_console.txt).

In [ ]:
#| eval: false
# open* within 5 words of data, beside share near data and beside the research terms. Writes open_terms.csv.
def opn(db, n=5):
    """A form of open within n words of data, in titles and abstracts."""
    if db == "PubMed":
        return "(" + " OR ".join(f'"{w} data"[tiab:~{n}]' for w in ("open", "opened", "opening", "openly", "openness")) + ")"
    return f"TITLE-ABS-KEY(open* W/{n} data)" if db == "Scopus" else f"TS=(open* NEAR/{n} data)"


def rs(db):
    return field(db, "anywhere", ["research*", "scienti*"])


def minus(db, a, b):
    return f"({a} AND NOT {b})" if db == "Scopus" else f"({a} NOT {b})"


B = [("(share NEAR/5 data) AND (research* OR scienti*)", lambda db: f"({share(db)} AND {rs(db)})"),
     ("(share NEAR/5 data OR open* NEAR/5 data) AND (research* OR scienti*)",
      lambda db: f"(({share(db)} OR {opn(db)}) AND {rs(db)})"),
     ("(share NEAR/5 data) AND (research* OR scienti* OR open* NEAR/5 data)",
      lambda db: f"({share(db)} AND ({rs(db)} OR {opn(db)}))")]
rows = []
for db in DATABASES:
    dev = [s for s in DEV if sl.testable(db, s)]
    test = [s for s in TEST if sl.testable(db, s)]
    base, base_q = None, B[0][1](db)
    for name, make in B:
        q = make(db)
        n = sl.count(db, f"({q}) AND {limit_near(db)}")
        d, t = sl.retrieved(db, q, dev), sl.retrieved(db, q, test)
        added = 0 if base is None else sl.count(db, f"({minus(db, q, base_q)}) AND {limit_near(db)}")
        base = d | t if base is None else base
        rows.append(dict(database=db, search=name, query=q, limit=limit_near(db), records_2015_2025=n,
                         records_added_to_query=added, development_found=len(d), development_of=len(dev),
                         test_found=len(t), test_of=len(test),
                         found_beyond_query="; ".join(sorted(LABEL.get(r, r) for r in (d | t) - base)),
                         lost_against_query="; ".join(sorted(LABEL.get(r, r) for r in base - d - t))))
    sl.write("open_terms.csv", rows)

##### Exploratory run 30. Proximity windows

**Question.** At what distance from *data* should *share* and *open\** sit?

**Measured.** (share NEAR/*n* data OR *open\** NEAR/*m* data) AND (*research\** OR *scienti\**), and (share NEAR/*n* data) AND (*research\** OR *scienti\**), for *n* and *m* of one, three, five and 10 words, 20 searches in each database. For each, the records for 2015 to 2025 and the development and test sources it finds and loses against five words for both.

**Result.** With the same distance for both terms, each cell gives records and sources lost against five words.

| Distance | PubMed | Scopus | Web of Science |
|:--|:--|:--|:--|
| One word | 8,564 · 0 | 31,571 · 4 | 22,477 · 4 |
| Three words | 11,995 · 0 | 45,786 · 2 | 33,168 · 2 |
| Five words | 14,371 · 0 | 55,738 · 0 | 40,738 · 0 |
| 10 words | 19,171 · 0 | 75,921 · 0 | 55,806 · 0 |

At three words, Scopus and Web of Science lose Costello (2009) and McCullough (2009); at one word, also Chen and Wu (2017) and Haddow et al. (2011). Ten words adds 40,051 records and no source. The file holds every pair of distances.

**Decision.** Five words for both. A distance of five words is sufficient for phrasings such as "sharing of research data" and "openly available data".

Files: [proximity_windows.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/proximity_windows.csv), [proximity_windows_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/proximity_windows_console.txt).

In [ ]:
#| eval: false
# share and open* at distances of 1, 3, 5 and 10 words from data, in every pair. Writes proximity_windows.csv.
def query_at(db, s, o):
    first = share_near_data(db, "anywhere", s) if o is None else f"({share_near_data(db, 'anywhere', s)} OR {opn(db, o)})"
    return f"({first} AND {rs(db)})"


WINDOWS = [1, 3, 5, 10]
rows = []
for db in DATABASES:
    dev = [s for s in DEV if sl.testable(db, s)]
    test = [s for s in TEST if sl.testable(db, s)]
    ref = sl.retrieved(db, query_at(db, 5, 5), dev) | sl.retrieved(db, query_at(db, 5, 5), test)
    for s in WINDOWS:
        for o in [None] + WINDOWS:
            name = f"(share NEAR/{s} data{'' if o is None else f' OR open* NEAR/{o} data'}) AND (research* OR scienti*)"
            q = query_at(db, s, o)
            n = sl.count(db, f"({q}) AND {limit_near(db)}")
            d, t = sl.retrieved(db, q, dev), sl.retrieved(db, q, test)
            rows.append(dict(database=db, search=name, share_window=s, open_window=o or "", query=q,
                             limit=limit_near(db), records_2015_2025=n, development_found=len(d),
                             development_of=len(dev), test_found=len(t), test_of=len(test),
                             found_beyond_5_5="; ".join(sorted(LABEL.get(r, r) for r in (d | t) - ref)),
                             lost_against_5_5="; ".join(sorted(LABEL.get(r, r) for r in ref - d - t))))
        sl.write("proximity_windows.csv", rows)

##### Exploratory run 31. The query against the test set, and its records

**Question.** How much of the test set does the query find, and how many records does it return?

**Measured.** The query against the test sources, checked by DOI or PMID in each database, with a Wilson 95% interval, as in exploratory run 26. Every record it returns for 2015 to 2025, with each record's DOI, title and year; records that share a DOI, or a title and year, count as one, as in exploratory run 27.

**Result.**

| | Test sources found | 95% interval | Records | With a DOI |
|:--|--:|:--|--:|--:|
| PubMed | 9 of 11 | 0.52 to 0.95 | 14,371 | 14,129 |
| Scopus | 20 of 23 | 0.68 to 0.96 | 55,738 | 51,871 |
| Web of Science | 13 of 18 | 0.49 to 0.88 | 40,738 | 37,418 |
| At least one database | 21 of 25 | 0.65 to 0.94 | 110,847 | |
| Unique | | | 60,852 | |

: Test sources the query finds, and the records it returns for 2015 to 2025. {tbl-colwidths="[28,18,18,18,18]"}

The query misses four test sources the databases index. Chiware and Mathe (2016) and Wittenberg and Elings (2017) concern research data management, Watson (2015) concerns open science, and Houtkoop et al. (2018) names neither a research term nor *open\** near *data*. Six test sources are in none of the databases or have no DOI or PMID.

Files: [test_coverage_open.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/test_coverage_open.csv), [test_missed_open.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/test_missed_open.csv), [open_query_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/open_query_records.csv), [open_query_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/open_query_console.txt).

In [ ]:
#| eval: false
# The query against the test set, as in exploratory run 26, and every record it returns, as in exploratory run 27.
# Writes test_coverage_open.csv, test_missed_open.csv and open_query_records.csv.
def query(db):
    return query_at(db, 5, 5)


found, indexed, rows = {}, {}, []
for db in DATABASES:
    t = [s for s in TEST if sl.testable(db, s)]
    indexed[db] = {s["ref_id"] for s in t if sl.count(db, sl.id_clause(db, [s])) > 0}
    found[db] = sl.retrieved(db, query(db), t)
    p, lo, hi = sl.wilson(len(found[db]), len(indexed[db]))
    rows.append(dict(database=db, query=query(db), limit=limit_near(db), found=len(found[db]),
                     indexed=len(indexed[db]), coverage=p, low=lo, high=hi))
anyf, inany = set().union(*found.values()), set().union(*indexed.values())
p, lo, hi = sl.wilson(len(anyf), len(inany))
rows.append(dict(database="all three", found=len(anyf), indexed=len(inany), coverage=p, low=lo, high=hi))
missed = []
for s in sorted(TEST, key=lambda s: s["label"]):
    if s["ref_id"] in anyf:
        continue
    where = {db: ("no DOI or PMID" if not (s["doi"] or s["pmid"]) else
                  "not indexed" if s["ref_id"] not in indexed[db] else "indexed, not found") for db in DATABASES}
    missed.append(dict(ref_id=s["ref_id"], label=s["label"], title=s["title"], year=s["year"], doi=s["doi"],
                       pmid=s["pmid"], field=s["field"], source_type=s["source_type"], pubmed=where["PubMed"],
                       scopus=where["Scopus"], web_of_science=where["Web of Science"]))
sl.write("test_coverage_open.csv", rows)
sl.write("test_missed_open.csv", missed)

recs = []
for db in DATABASES:
    recs += [dict(r, database=db) for r in fetch(db, query(db), limit_near(db))]
    sl.write("open_query_records.csv", recs)
print(len(recs), unique_count(recs))

### Rerunning the supplement

To repeat its work, run the first code cell below to build `reference_sources_coded.csv` from the local model's codes. Before any database run, run the setup cell under Open data and research data management and then the exploratory setup cell.

Every file comes from code in the repository except one judgment, the comparison of Chawinga and Zinn's criteria with RDS-Review's in `comparison.csv`.

| Step | Command |
| --- | --- |
| Coding, after the local model is running | `LLM_URL=http://<GPU server>:11450 python3 analysis/00_search-development/local-model/coding.py reference_codes_qwen3.5-122b.csv 2>&1 \| tee $O/coding_console_qwen3.5-122b.txt` |
| Scope sample, after the local model is running | `LLM_URL=http://<GPU server>:11450 python3 analysis/00_search-development/local-model/scope.py scope_screening_qwen3.5-122b.csv 2>&1 \| tee $O/scope_screening_console_qwen3.5-122b.txt` |


:::